# The power of workflow orchestration: a Circle of Willis cohort as a tensor DAG

*medical · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/cow_workflow_dag)

A vascular study is a chain of steps (segment the MRA, extract the centreline graph, measure the vessels, place the inlets and outlets, solve the flow, report), each already a tensor program, so the workflow that Snakemake orchestrates can be a tensor too. On six open (CC0) TopCoW MRA scans of the Circle of Willis, the same Snakefile is run by Snakemake (offline) and by a tensor DAG engine: the job graph is a sum of Kronecker products, its levels, critical path and rerun set are matrix–vector iterations, and the results match job for job. Then the time saved, where it comes from, and what a file DAG cannot give: a gradient through the whole pipeline. Educational only, not for clinical use.

**Vector form**

$$A = \sum_{s,r} (A_{\mathrm{rule}})_{sr}\, E_{sr} \otimes \bigotimes_a M_a^{sr},\qquad (I - A)^{-1} = \sum_{k \ge 0} A^k,\qquad t = d \oplus (A \otimes t),\qquad K\,p = b,\quad G_e = \frac{\pi r_e^4}{8\mu L_e}$$

**Index (tensor) form**

$$M_a^{sr} \in \{I_{n_a},\ \mathbb 1_{n_a},\ \mathbb 1_{n_a}^{\top}\},\qquad \operatorname{tr}(A^k) = 0\ \ (k = 1 \dots J),\qquad \ell_i = \max_j A_{ij}(\ell_j + 1),\qquad t_i = d_i + \max_j A_{ij}\, t_j,\qquad x = s \lor R\,s \qquad K_{nm} = \sum_e G_e B_{en} B_{em} + g_n \delta_{nm},\quad g_k = \frac{Q\, r_k^3}{P \sum_m r_m^3},\quad r_e = \sqrt{\frac{h^3 \sum_{x \to e} \sigma\big((I_x - \theta)/w\big)}{\pi L_e}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{\mathrm{rule}}$ | `rule_graph(parse_snakefile(text))` | (R, R) | rule $s$ reads a file pattern that rule $r$ writes; parsed from the bundled Snakefile itself |
| $M_a^{sr}$ | `kron_block(axes_r, axes_s, sizes)` | (n_s, n_r) | per wildcard axis $a$ (case, condition): $I$ when both rules carry it (per-case edges $A \otimes I$), $\mathbb 1$ for a broadcast, $\mathbb 1^\top$ for a gather such as the cohort report |
| $A$ | `job_adjacency(A_rule, axes, sizes)` | (J, J) | $A_{ij} = 1$ when job $i$ reads an output of job $j$; 50 jobs, 79 edges |
| $\operatorname{tr}(A^k)$ | `walk_traces(A), is_nilpotent(A)` | (J,) | closed walks of length $k$; all zero ⇔ $A$ nilpotent ⇔ acyclic |
| $(I - A)^{-1}$ | `torch.linalg.inv(I - A), closure(A)` | (J, J) | counts the paths $j \to i$; its 0/1 pattern is the reachability (boolean repeated squaring) |
| $\ell_i$ | `levels(A)` | (J,) | topological level by boolean matrix–vector iteration: a job is ready when $A\,\neg\mathrm{done} = 0$ in its row |
| $t = d \oplus (A \otimes t)$ | `finish_times(A, d)` | (J,) | earliest finish times, a max-plus (tropical) fixed point; $\max_i t_i$ is the critical path |
| $x = s \lor R s$ | `rerun_set(closure(A), touched)` | (J,) | jobs to rerun after the jobs in $s$ (readers of a changed file) became outdated |
| $\sigma((I - \theta)/w)$ | `soft_threshold(image, theta, width)` | (B, X, Y, Z) | soft segmentation, differentiable in the threshold $\theta$ (one per case) |
| $T_v = \min_u (T_u + \ell_{uv}/R_{uv}^2)$ | `relax(T, src, dst, w)` | (V,) | min-plus eikonal on the voxel graph of all vessels of all cases at once: free ends and centrelines |
| $r_e$ | `measure(seg, g)['radius']` | (B, E) | edge radius from the soft volume of the voxels nearest to its centreline |
| $K\,p = b$ | `torch.linalg.solve(K, b)` | (B, N, N) | Kirchhoff with Poiseuille conductances $G_e$ and Murray outlet loads $g_k$, one batched solve; Dirichlet rows $p = P$ at the open inlets |
| $\partial s/\partial\theta$ | `share.sum().backward()` | (B,) | the left-MCA flow share differentiated through flow, measure and segmentation, every case in one backward pass |

Idea. The framing, the power of workflow orchestration, is Jacky Ko's, from his HPC and PhD work on vascular pipelines: if every stage of the pipeline is a tensor program, the DAG that a workflow manager schedules can be one too. This problem builds that and measures it. Workflow. data/Snakefile has eight rules (segment, centreline, measure, boundaries, flow, report, cohort, all) and two wildcards: the case (six scans) and the condition (both ICA inlets open, or the left ICA closed). Each rule's shell runs workflow.py, which calls the same tensor function for one case that the tensor engine calls once for all cases. Snakemake 9.27.1 ran it offline in WSL2 (its own environment, never part of this project) and its outputs are bundled as references in data/snakemake: the job DAG (--dag), the dry-run job list, the rerun set after one input changed, --summary, the per-job start and end times, and the cohort table its jobs wrote. The tests check that the tensor engine has the same 50 jobs and 79 edges, that Snakemake's orders respect our $A$ and give our levels, that the rerun set is the same, and that the cohort table is identical to the 9 digits written. A detail on reruns: touching an input did not make Snakemake rerun anything, because for small files it compares checksums; the reference changes the file's bytes (same arrays), and then 10 jobs rerun, as $x = s \lor Rs$ predicts. Time. In the bundled run Snakemake took 118 s on 1 core, 43 s on 6 and 27 s on 12 (an earlier run on a less busy machine: 79, 23 and 22 s); the tensor engine runs the same work in about 5 to 10 s on one thread. The honest accounting: each Snakemake job is a new process that starts Python and imports torch and the job module (about 1.8 s, data/snakemake/startup.csv, so about 88 s of the 115 s of summed job time), reads and writes .npz files, and is scheduled; the tensor engine pays that once. Batching itself buys little on a CPU here (running the cases one at a time in-process takes about as long, sometimes less), so the gain comes from start-up and files, and it shrinks when every job is a heavy GPU-bound solve that dwarfs the start-up. No schedule on $c$ cores beats $\max(\mathrm{CP}, W/c)$, the critical path from the max-plus iteration and the total work over $c$, and the measured runs respect it; finding the best schedule is NP-hard in general (Graham's list scheduling is within $W/c + \mathrm{CP}$). Gradient. A file DAG passes numbers, not derivatives. The soft threshold makes segmentation, radii and flow differentiable, so $\partial s_{\mathrm{L\,MCA}}/\partial\theta$ (the left-MCA share of the outflow) comes for every case from one backward pass and matches central finite differences, which would cost Snakemake two extra full runs. The centreline graph (topology) and the connected-component mask are held fixed. Pipeline. Segment: soft threshold $\sigma((I - \theta)/w)$ of the normalised TOF intensity, hard mask $I > \theta$ minus components under 64 voxels, Dice against the TopCoW mask about 0.86 to 0.91. Centreline: from the TopCoW vessel labels, junctions where labelled vessels meet (contacts within 3.5 mm merged; only anatomical pairs count, so two A2s running side by side do not), a free end per terminal vessel (its farthest voxel by a min-plus eikonal with travel time $\ell/R^2$, re-centred), and the shortest centred path between consecutive nodes (as in the aneurysm problem's tensor stand-in for VMTK). Measure: each voxel near the labelled vessels goes to its nearest centreline point, so an edge's radius is $\sqrt{V_e/(\pi L_e)}$ from the soft volume; asymmetry $2(r_R - r_L)/(r_R + r_L)$ per vessel; a stenosis flag where the smoothed radius profile dips below half its median (in these healthy controls such flags are segmentation artefacts). Boundaries, as in Venova: inlets at the free ends of the ICAs and the basilar artery, outlets at the MCA, ACA and PCA ends; Pcom and Acom ends are never boundaries. Flow: a reduced-order steady model, Poiseuille conductance per edge, $P$ = 90 mmHg at the open inlets, each outlet drained to 0 Pa through a Murray load $g_k \propto r_k^3$ tuned so that a resistance-free network would carry 10 ml/s; one batched linear solve. The collateral test closes the left ICA: where an Acom or a Pcom is patent, the left hemisphere stays perfused. Simplified: Newtonian blood, rigid straight tubes, no pulsatility, no 3D flow (the aneurysm problem does 3D CFD on one vessel), threshold radii on 0.5 mm voxels, literature-order pressures and flows. Data. TopCoW 2024, MRA_Lausanne subset (Zenodo 10.5281/zenodo.15692630, licence CC0): 20 healthy-control TOF-MRA scans with Circle of Willis labels. Six cases are bundled as 96×80×48 crops at 0.5 mm (image and labels, under 1 MB in all); data/SOURCES.md has the processing and the citations (Yang et al., the TopCoW challenge; Di Noto et al. 2022, the Lausanne cohort). The main TopCoW release is not used (its licence restricts commercial use). Results. Besides the GIF there are two 3D results. results/centreline.pvd is the anatomy: every case's segmentation surface (marching tetrahedra) and its centreline graph, both coloured by the vessel radius. results/flow.pvd is the flow of the 1D Poiseuille network over 11 steps in which the left ICA narrows: the timestep is the stenosis severity s, the fraction of the lumen diameter lost (0 open, 1 closed). Its layers are the vessel surface coloured by the wall shear stress $\tau_w = 4\mu|Q|/(\pi r^3)$ of the nearest centreline edge (pressure as a second array), the streamlines and particles (both coloured by speed), and the inlet and outlet discs with arrows. There is no 3D CFD here: the streamlines are the idealised axial flow of the 1D model, lines parallel to each vessel's centreline at $\rho = 0, 0.35, 0.7$ of its radius with the Hagen–Poiseuille speed $u = 2U(1 - \rho^2)$, $U = |Q|/(\pi r^2)$; there is no secondary flow, and they break at the junctions. Each step is a steady state; the particles only advance along their streamlines by a fixed pseudo-time of 4 ms per step (wrapping round to the start of their vessel) to show the speeds. The stenosis is a 10 mm Poiseuille segment in series with the inlet, so its conductance falls as $(1 - s)^4$; s = 0 and s = 1 are exactly the workflow's two conditions, and the steps between are run by the tensor engine only (one batched solve per step). results/stenosis_sweep.csv gives the left-ICA inflow and left-MCA share per step and case; results/timings.csv, results/equivalence.csv and results/cohort.csv hold the other numbers.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/cow_workflow_dag/main.py
"""Main loop: read the Snakefile, build the job DAG as tensors, check it against the bundled Snakemake run (jobs,
edges, levels, order, rerun set, outputs), run the whole cohort as tensors (one batched call per rule), compare the
wall-clock times, differentiate the flow split with respect to the segmentation threshold, then draw and export."""

import csv
import io
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F

from problems._common.runtime import cli, fast_mode

from . import equation as eq
from . import plot, solve
from . import tensors as T


def equivalence(dag: dict, cohort: str | None) -> list[dict]:
    """The tensor engine against the bundled Snakemake references, one row per check."""
    idx = {j: i for i, j in enumerate(dag["jobs"])}
    A = dag["A"]
    rows = []
    jobs_ref, edges_ref, _ = T.ref_dag()
    ours = {(dag["jobs"][i], dag["jobs"][j]) for i, j in A.nonzero().tolist()}
    rows.append({"check": "jobs", "tensor": len(dag["jobs"]), "snakemake": len(jobs_ref),
                 "ok": set(dag["jobs"]) == jobs_ref})  # fmt: skip
    rows.append({"check": "edges", "tensor": len(ours), "snakemake": len(edges_ref), "ok": ours == edges_ref})
    traces = eq.walk_traces(A)
    rows.append({"check": "acyclic", "tensor": f"max {int(traces.abs().max())}", "snakemake": "a DAG",
                 "ok": eq.is_nilpotent(A)})  # fmt: skip
    lev = eq.levels(A)
    order = [idx[j] for j in T.ref_jobs("dryrun_jobs.csv")]
    pos = {k: n for n, k in enumerate(order)}
    valid = all(pos[i] > pos[j] for i, j in A.nonzero().tolist())
    rows.append({"check": "order, levels", "tensor": f"{int(lev.max()) + 1} levels",
                 "snakemake": f"{len(order)} jobs", "ok": valid})  # fmt: skip
    touched = torch.zeros(len(A), dtype=torch.bool)
    touched[[idx[("segment", "sub-002", "")], idx[("centreline", "sub-002", "")]]] = True
    x = eq.rerun_set(eq.closure(A), touched)
    ref = set(T.ref_jobs("rerun_after_touch.csv"))
    mine = {dag["jobs"][k] for k in x.nonzero().flatten().tolist()}
    rows.append({"check": "rerun set", "tensor": len(mine), "snakemake": len(ref), "ok": mine == ref})
    if cohort is not None:
        a, b = list(csv.DictReader(io.StringIO(cohort))), T.ref_cohort()
        worst = 0.0
        for ra, rb in zip(a, b):
            for k in solve.COLUMNS[2:]:
                x1, x2 = float(ra[k]), float(rb[k])
                if not (math.isnan(x1) and math.isnan(x2)):
                    worst = max(worst, abs(x1 - x2) / max(abs(x2), 1e-9))
        rows.append({"check": "cohort table", "tensor": f"{len(a)} rows",
                     "snakemake": f"Δ {worst:.0e}", "ok": len(a) == len(b) and worst < 1e-6})  # fmt: skip
    return rows


def run(params: dict, out_dir: Path) -> Path:
    torch.manual_seed(0)
    theta = float(params["threshold"])
    cores = int(params["cores"])
    cases = T.CASES
    t_all = time.perf_counter()

    # 1. the Snakefile → rule graph → job DAG (Kronecker) and its algebra
    dag = solve.build_dag(T.SNAKEFILE.read_text(encoding="utf-8"))
    names = [r["name"] for r in dag["rules"]]
    A, A_rule = dag["A"], dag["A_rule"]
    print(f"Snakefile: {len(names)} rules {names}; wildcards case × condition = {len(cases)} × {len(T.CONDITIONS)}")
    print("rule edges (consumer ← producer): " + ", ".join(f"{names[s]}←{names[r]}" for s, r in
                                                          A_rule.nonzero().tolist()))  # fmt: skip
    print(f"job DAG: A = Σ E_sr ⊗ (⊗_a M_a): {len(dag['jobs'])} jobs, {int(A.sum())} edges;  "
          f"nilpotent (A^J = 0): {eq.is_nilpotent(A)};  "
          f"max |tr(A^k)| = {int(eq.walk_traces(A).abs().max())}")  # fmt: skip
    lev = eq.levels(A)
    print("levels: " + ", ".join(f"ℓ={k}: {int((lev == k).sum())} jobs" for k in range(int(lev.max()) + 1)))
    paths = torch.linalg.inv(torch.eye(len(A), dtype=torch.float64) - A.double())
    print(f"(I − A)⁻¹ counts paths: {int(paths.sum().round())} job pairs × paths in total, "
          f"{int((paths.round() > 0).sum()) - len(A)} reachable pairs")  # fmt: skip

    # 2. run the cohort as tensors: one batched call per rule (single thread, like each Snakemake job)
    threads = torch.get_num_threads()
    torch.set_num_threads(1)
    store, rule_times = solve.run_engine(dag, cases, T.CONDITIONS, theta)
    t_engine = max(b for _, b in rule_times.values())
    t0 = time.perf_counter()
    for c in cases:  # in-process, one case at a time (no process start-up, no files): isolates batching
        solve.run_engine(dag, (c,), T.CONDITIONS, theta)
    t_percase = time.perf_counter() - t0
    torch.set_num_threads(threads)
    print("tensor engine, rule by rule (one batched call each): " + ", ".join(
        f"{k} {b - a:.2f} s" for k, (a, b) in rule_times.items() if b - a > 0.005))  # fmt: skip
    print(f"tensor engine: {t_engine:.2f} s for {len(dag['jobs'])} jobs;  in-process one case at a time: "
          f"{t_percase:.2f} s (1 thread)")  # fmt: skip

    # 3. equivalence with Snakemake
    equiv = equivalence(dag, store["cohort"] if abs(theta - T.THETA) < 1e-12 else None)
    for r in equiv:
        print(f"  {'✓' if r['ok'] else '✗'} {r['check']}: tensor {r['tensor']}, Snakemake {r['snakemake']}")
    if abs(theta - T.THETA) >= 1e-12:
        print(f"  (cohort table not compared: the Snakemake run used θ = {T.THETA})")

    # 4. time: Snakemake's measured runs and the scheduling bounds from its job times
    idx = {j: i for i, j in enumerate(dag["jobs"])}
    walls = T.ref_runs()
    d = torch.zeros(len(A), dtype=torch.float64)
    for r in T.ref_timings(1):
        d[idx[(r["rule"], r["case"], r["condition"])]] = r["end"] - r["start"]
    t_fin = eq.finish_times(A, d)
    cp = float(t_fin.max())
    crit = eq.critical_path(A, t_fin, d)
    W = float(d.sum())
    print(f"Snakemake (measured, WSL2, 1 thread per job): 1 core {walls[1]:.1f} s, 6 cores {walls[6]:.1f} s, "
          f"12 cores {walls[12]:.1f} s")  # fmt: skip
    print(f"from its 1-core job times: work W = {W:.1f} s, critical path (max-plus) = {cp:.1f} s through "
          + " → ".join(f"{dag['jobs'][k][0]}" for k in crit))  # fmt: skip
    for c in sorted({1, 6, 12, cores}):
        print(f"  c = {c:2d}: lower bound max(CP, W/c) = {eq.lower_bound(d, cp, c):.1f} s"
              + (f", measured {walls[c]:.1f} s" if c in walls else ""))  # fmt: skip
    lines = (T.REF / "startup.csv").read_text().splitlines()
    startup = [float(x.split(",")[1]) for x in lines if x.startswith("import_job_module")]
    st = sum(startup) / len(startup)
    n_jobs = len(dag["jobs"]) - 1  # the target rule 'all' starts no process
    print(f"per-job overhead: starting Python and importing torch and the job module takes {st:.2f} s, × {n_jobs} "
          f"jobs = {st * n_jobs:.0f} s of the {W:.0f} s of work; the rest is file I/O and the compute "
          "itself")  # fmt: skip
    print(f"speed-up of the tensor engine: ×{walls[1] / t_engine:.1f} over 1 core, ×{walls[12] / t_engine:.1f} over "
          f"12 cores. It comes from start-up and files, not from batching (one case at a time in-process: "
          f"{t_percase:.1f} s); for a heavy GPU solve per job the start-up share, and the gain, shrink.")  # fmt: skip

    # 5. the gradient: d(left-MCA share)/dθ for every case in one backward pass, against finite differences
    img, lab, g, bnd = store["image"], store["labels"], store["centreline"], store["boundaries"]
    keep, near = store["segment"]["keep"], store["measure"]["near"]
    occ = torch.zeros(len(cases), dtype=torch.bool)

    def share(th):
        seg = solve.segment(img, lab, th, keep=keep)
        m = solve.measure(seg, g, near=near)
        return solve.report(m, bnd, g, solve.flow(m, bnd, g, occ))["share_L-MCA"]

    th = torch.full((len(cases),), theta, dtype=torch.float64, requires_grad=True)
    t0 = time.perf_counter()
    s = share(th)
    s.sum().backward()  # cases are independent, so one backward pass gives every ∂s_b/∂θ_b
    t_grad = time.perf_counter() - t0
    dth = 2e-3
    fd = (share(th.detach() + dth) - share(th.detach() - dth)) / (2 * dth)
    print(f"∂(L-MCA share)/∂θ, one backward pass ({t_grad:.2f} s): "
          + ", ".join(f"{c} {a:+.3f} (FD {b:+.3f})" for c, a, b in zip(cases, th.grad.tolist(), fd.tolist()))
          + f";  max |autograd − FD| = {float((th.grad - fd).abs().max()):.1e}  "
          "(Snakemake: 2 extra full reruns)")  # fmt: skip

    # 6. the clinical-style numbers per case
    rep = {k: v.detach().reshape(len(cases), -1) for k, v in store["report"].items()}
    for k, c in enumerate(cases):
        print(f"{c}: Dice {rep['dice'][k, 0]:.3f}, {int(rep['nodes'][k, 0])} nodes, {int(rep['edges'][k, 0])} "
              f"edges, {rep['length_mm'][k, 0]:.0f} mm of centreline;  inflow R-ICA/L-ICA/BA "
              f"{rep['Q_in_R-ICA'][k, 0]:.2f}/{rep['Q_in_L-ICA'][k, 0]:.2f}/{rep['Q_in_BA'][k, 0]:.2f} ml/s, "
              f"Δp {rep['dp_max_mmHg'][k, 0]:.1f} mmHg;  L-ICA closed: L-MCA "
              f"{100 * rep['Q_L-MCA'][k, 1] / rep['Q_L-MCA'][k, 0]:.0f} %, L-ACA "
              f"{100 * rep['Q_L-ACA'][k, 1] / max(float(rep['Q_L-ACA'][k, 0]), 1e-9):.0f} % of baseline;  "
              f"stenosis flags {int(rep['stenoses'][k, 0])}")  # fmt: skip

    # 7. results: tables, the 3D scene, the animation
    res = out_dir / "results"
    res.mkdir(parents=True, exist_ok=True)
    (res / "cohort.csv").write_text(store["cohort"], encoding="utf-8")
    with open(res / "equivalence.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["check", "tensor", "snakemake", "ok"])
        for r in equiv:
            w.writerow([r["check"], r["tensor"], r["snakemake"], r["ok"]])
    bars = {"Snakemake 1 core": walls[1], "Snakemake 6 cores": walls[6], "Snakemake 12 cores": walls[12],
            "bound, 12 cores": eq.lower_bound(d, cp, 12), "case by case": t_percase,
            "tensor engine": t_engine}  # fmt: skip
    with open(res / "timings.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["# wall-clock seconds for the whole cohort; Snakemake measured offline (WSL2, AMD Ryzen 5 5600G, "
                    "1 thread per job), the tensor engine measured in this run (1 thread)"])  # fmt: skip
        w.writerow(["how", "seconds"])
        for k, v in bars.items():
            w.writerow([k, f"{v:.3f}"])
        for name, (a, b) in rule_times.items():
            w.writerow([f"tensor rule {name}", f"{b - a:.4f}"])
    surfaces = []
    for k in range(len(cases)):  # marching tetrahedra of the soft segmentation inside the labelled-vessel region
        field = (store["segment"]["soft"][k] * store["segment"]["region"][k]).detach()
        field = F.avg_pool3d(field[None, None], 3, 1, 1)[0, 0]
        surfaces.append(tuple(x.numpy() for x in solve.iso_surface(field, 0.5, T.H)))
    for old in res.glob("*.vtp"):  # stale pieces of an earlier layout
        old.unlink()
    (res / "cow.pvd").unlink(missing_ok=True)
    # 6. a longer flow sequence for the 3D result: the left ICA narrows from open (s = 0) to closed (s = 1)
    with torch.no_grad():
        sev, sweep = solve.stenosis_sweep(store["measure"], bnd, g)
    l_mca = [solve.report(store["measure"], bnd, g, fs)["share_L-MCA"] for fs in sweep]
    l_ica = [solve.report(store["measure"], bnd, g, fs)["Q_in_L-ICA"] for fs in sweep]
    with open(res / "stenosis_sweep.csv", "w", newline="", encoding="utf-8") as f:
        f.write("# Left-ICA stenosis sweep: severity s = fraction of the lumen diameter lost (0 open, 1 closed).\n")
        f.write("# Steady 1D network flow per case (tensor engine only; the Snakemake workflow runs s = 0 and 1).\n")
        w = csv.writer(f)
        w.writerow(["severity", "case", "Q_in_L-ICA_ml_s", "share_L-MCA"])
        for i, s in enumerate(sev.tolist()):
            for k, c in enumerate(cases):
                w.writerow([f"{s:.2f}", c, f"{float(l_ica[i][k]):.4f}", f"{float(l_mca[i][k]):.4f}"])
    print("left-ICA stenosis sweep (" + ", ".join(f"s={s:.2f}" for s in sev[::5].tolist()) + "): mean left-ICA "
          "inflow " + " → ".join(f"{float(x.mean()):.2f}" for x in l_ica[::5]) + " ml/s")  # fmt: skip
    anatomy, flow = plot.write_scene(res, store, cases, surfaces, sev, sweep)
    size = sum(f.stat().st_size for f in res.iterdir() if f.suffix == ".vtp") / 1e6
    wss = [eq.wall_shear_stress(fs["q"], store["measure"]["radius"], T.MU)[g["edge_valid"]] for fs in sweep]
    print(f"3D results: {anatomy.relative_to(out_dir)} (segmentation and centrelines by radius) and "
          f"{flow.relative_to(out_dir)} over {len(sev)} steps of left-ICA stenosis (s = 0 open → 1 closed): surface "
          f"WSS = 4μ|Q|/(πr³) (and pressure), idealised axial Poiseuille streamlines and particles "
          f"(Δt = {plot.DT * 1e3:.0f} ms of pseudo-time per step), inlets/outlets; median WSS "
          f"{float(wss[0].median()):.2f} Pa at s = 0, {float(wss[-1].median()):.2f} Pa at s = 1;  "
          f"{sum(len(s[1]) for s in surfaces)} surface triangles before decimation, {size:.1f} MB")  # fmt: skip
    sm12 = T.ref_timings(12)
    timing = {"levels": lev.numpy(), "critical": crit, "snakemake12": sm12, "wall12": walls[12], "engine": rule_times,
              "end12": {(r["rule"], r["case"], r["condition"]): r["end"] for r in sm12}, "bars": bars}  # fmt: skip
    grad = {"autograd": th.grad.numpy(), "fd": fd.numpy()}
    print(f"compute {time.perf_counter() - t_all:.1f} s (animation next{', fast mode' if fast_mode() else ''})")
    return plot.animate(store, dag, list(cases), timing, equiv, grad, out_dir / "cow_workflow_dag.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/cow_workflow_dag/equation.py
r"""The laws as tensor operations: a workflow DAG in linear algebra, and the vessel-network laws.

Workflow (jobs i, j; rules r, s; A_ij = 1 when job i reads a file that job j writes):

    job DAG          A = Σ_{s,r} (A_rule)_{sr} E_{sr} ⊗ M^{sr},  M^{sr} = ⊗_a M_a   (I, 𝟙 or 𝟙ᵀ per wildcard axis a)
    acyclic          A nilpotent ⇔ A^J = 0 ⇔ trace(A^k) = 0 for k = 1 … J (no closed walks)
    reachability     (I − A)^{-1} = I + A + A² + …  (a finite sum: it counts the paths j → i)
    levels           ℓ_i = max_j A_ij (ℓ_j + 1)  (boolean matrix–vector iteration)
    makespan         t_i = d_i + max_j A_ij t_j   (max-plus, i.e. tropical, matrix–vector iteration)
    rerun set        x = s ∨ (R s),  R = boolean closure of A, s = jobs that read a touched file

Vessel network (edges e = (u, v), nodes n, repeated indices summed):

    Poiseuille       G_e = π r_e⁴ / (8 μ L_e)
    Murray outlets   g_k = Q_tot r_k³ / (P_in Σ_m r_m³)
    Kirchhoff        K_nm p_m = b_n,  K = Bᵀ diag(G) B + diag(g)   (B the (E, N) incidence matrix)
    axial profile    u(ρ) = 2U(1 − ρ²),  U = |Q_e|/(π r_e²),  ρ = distance from the axis / r_e
    wall shear       τ_w = μ |∂u/∂n|_wall = 4μ|Q_e|/(π r_e³)
    soft threshold   s(x) = σ((I(x) − θ)/w),  differentiable in θ
"""

import math

import torch


# ---------------------------------------------------------------- the workflow DAG as tensors
def kron_block(axes_r: tuple, axes_s: tuple, sizes: dict) -> torch.Tensor:
    r"""Job-level block M^{sr} (n_s, n_r) between a producer rule r and a consumer rule s: a Kronecker product over the
    wildcard axes, I when both rules carry the axis (per-case edges A ⊗ I), 𝟙 (n_a, 1) when only the consumer does
    (a broadcast), 𝟙ᵀ (1, n_a) when only the producer does (a gather, e.g. into the cohort report)."""
    M = torch.ones(1, 1)
    for a, n in sizes.items():  # the canonical axis order: jobs are numbered row-major over these axes
        if a in axes_r and a in axes_s:
            M = torch.kron(M, torch.eye(n))
        elif a in axes_s:
            M = torch.kron(M, torch.ones(n, 1))
        elif a in axes_r:
            M = torch.kron(M, torch.ones(1, n))
    return M


def job_adjacency(A_rule: torch.Tensor, axes: list, sizes: dict) -> tuple[torch.Tensor, torch.Tensor]:
    r"""A = Σ_{s,r} (A_rule)_{sr} E_{sr} ⊗ M^{sr}: the (J, J) job adjacency and the offset of each rule's jobs."""
    n = torch.tensor([math.prod(sizes[a] for a in ax) for ax in axes])
    off = torch.cat([torch.zeros(1, dtype=torch.long), n.cumsum(0)])
    A = torch.zeros(int(off[-1]), int(off[-1]))
    for s, r in A_rule.nonzero().tolist():  # one block per rule edge (a handful), each a Kronecker product
        A[off[s] : off[s + 1], off[r] : off[r + 1]] = kron_block(axes[r], axes[s], sizes)
    return A, off


def walk_traces(A: torch.Tensor) -> torch.Tensor:
    r"""trace(A^k) for k = 1 … J: the number of closed walks of length k. All zero ⇔ no cycle ⇔ A nilpotent."""
    out, P = [], torch.eye(len(A), dtype=torch.float64)
    for _ in range(len(A)):
        P = P @ A.double()
        out.append(torch.trace(P))
    return torch.stack(out)


def is_nilpotent(A: torch.Tensor) -> bool:
    r"""A^J = 0 (checked by repeated squaring in the boolean semiring, so no overflow)."""
    P, k = A > 0, 1
    while k < len(A):
        P = (P.double() @ P.double()) > 0
        k *= 2
    return not bool(P.any())


def closure(A: torch.Tensor) -> torch.Tensor:
    r"""R = A + A² + … in the boolean semiring (R_ij: job i depends on job j, directly or not): repeated squaring
    R ← R ∨ R·R, about log₂(depth) products. (I − A)^{-1} = I + R as a 0/1 pattern."""
    R = A > 0
    while True:
        R2 = R | ((R.double() @ R.double()) > 0)
        if torch.equal(R2, R):
            return R
        R = R2


def levels(A: torch.Tensor) -> torch.Tensor:
    r"""ℓ_i = 0 for jobs with no inputs from other jobs, else 1 + max over their inputs: a job can start once every
    job of a lower level is done. Boolean iteration: done ← done ∨ (A·¬done = 0)."""
    done = torch.zeros(len(A), dtype=torch.bool)
    lev = torch.full((len(A),), -1, dtype=torch.long)
    k = 0
    while not bool(done.all()):
        ready = ~done & ((A.double() @ (~done).double()) == 0)  # all inputs done
        if not bool(ready.any()):
            raise ValueError("cycle: no job is ready")
        lev[ready], done = k, done | ready
        k += 1
    return lev


def finish_times(A: torch.Tensor, d: torch.Tensor) -> torch.Tensor:
    r"""Earliest finish times with unlimited cores: t_i = d_i + max_j A_ij t_j, a max-plus (tropical) fixed point
    reached after at most depth + 1 sweeps. max over an empty set = 0."""
    t = d.clone()
    for _ in range(len(A) + 1):
        start = torch.where(A > 0, t[None, :], torch.tensor(-math.inf, dtype=t.dtype)).max(1).values.clamp(min=0)
        t_new = d + start
        if torch.equal(t_new, t):
            return t
        t = t_new
    raise ValueError("no fixed point: the graph has a cycle")


def critical_path(A: torch.Tensor, t: torch.Tensor, d: torch.Tensor) -> list[int]:
    r"""The chain of jobs whose durations add up to the makespan: backtrack from argmax t through the input j that
    finishes exactly when job i starts (t_i − d_i)."""
    i = int(t.argmax())
    path = [i]
    while True:
        deps = A[i].nonzero().flatten()
        if len(deps) == 0:
            return path[::-1]
        i = int(deps[(t[deps] - (t[i] - d[i])).abs().argmin()])
        path.append(i)


def rerun_set(R: torch.Tensor, touched: torch.Tensor) -> torch.Tensor:
    r"""Jobs to rerun after the jobs in `touched` (bool (J,)) became outdated: x = s ∨ R s (boolean)."""
    return touched | ((R.double() @ touched.double()) > 0)


def lower_bound(d: torch.Tensor, cp: float, cores: int) -> float:
    r"""No schedule on c cores beats max(critical path, total work / c) (exact scheduling is NP-hard)."""
    return max(cp, float(d.sum()) / cores)


# ---------------------------------------------------------------- segmentation
def soft_threshold(image: torch.Tensor, theta: torch.Tensor, width: float) -> torch.Tensor:
    r"""s = σ((I − θ)/w) per voxel; image (B, X, Y, Z), theta (B,). Tends to the hard threshold I > θ as w → 0."""
    return torch.sigmoid((image - theta[:, None, None, None]) / width)


def dice(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    r"""Dice = 2|a ∩ b| / (|a| + |b|) per case; a, b (B, X, Y, Z) bool → (B,)."""
    inter = (a & b).flatten(1).sum(1).double()
    return 2 * inter / (a.flatten(1).sum(1) + b.flatten(1).sum(1)).double().clamp(min=1)


# ---------------------------------------------------------------- centreline: min-plus eikonal on the voxel graph
def edge_cost(length: torch.Tensor, R_mid: torch.Tensor, power: float = 2.0) -> torch.Tensor:
    r"""Travel time ℓ / R^p along a voxel edge: the front runs fastest along the middle of the vessel."""
    return length / R_mid.clamp(min=1e-9) ** power


def relax(T: torch.Tensor, src: torch.Tensor, dst: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""One min-plus (Bellman–Ford) sweep: T_v ← min(T_v, min_u T_u + w_uv) over the directed edges u → v."""
    return T.scatter_reduce(0, dst, T[src] + w, reduce="amin", include_self=True)


# ---------------------------------------------------------------- the vessel network
def poiseuille_conductance(r: torch.Tensor, L: torch.Tensor, mu: float) -> torch.Tensor:
    r"""G = πr⁴/(8μL) (m³ s⁻¹ Pa⁻¹): Hagen–Poiseuille flow Q = G Δp through a straight tube."""
    return math.pi * r**4 / (8 * mu * L.clamp(min=1e-9))


def poiseuille_speed(rho, U):
    r"""Hagen–Poiseuille axial speed u(ρ) = 2U(1 − ρ²) at the relative radius ρ = d/r ∈ [0, 1] (U the mean speed)."""
    return 2 * U * (1 - rho**2)


def wall_shear_stress(q, r, mu: float):
    r"""τ_w = μ |∂u/∂d| at the wall of a straight Poiseuille tube = 4μ|Q|/(πr³) (Pa), Q (m³/s) and r (m)."""
    return 4 * mu * abs(q) / (math.pi * r**3)


def murray_conductance(r_out: torch.Tensor, outlet: torch.Tensor, Q_tot: float, P: float) -> torch.Tensor:
    r"""Peripheral conductance of each outlet, g_k = Q_tot r_k³ / (P Σ_m r_m³): with a resistance-free network every
    outlet would take a share ∝ r³ (Murray's law) of Q_tot at the pressure P. r_out, outlet (B, N) → (B, N)."""
    r3 = torch.where(outlet, r_out, torch.zeros_like(r_out)) ** 3
    return Q_tot * r3 / (P * r3.sum(1, keepdim=True).clamp(min=1e-30))


def kirchhoff_matrix(G: torch.Tensor, edges: torch.Tensor, n_nodes: int) -> torch.Tensor:
    r"""K = Bᵀ diag(G) B, assembled from the edge list: K_uu += G, K_vv += G, K_uv −= G, K_vu −= G.
    G (B, E), edges (B, E, 2) → (B, N, N). (Q into node n) = −(K p)_n; mass balance is K p = b."""
    Bn, E = G.shape
    b = torch.arange(Bn)[:, None].expand(Bn, E)
    u, v = edges[..., 0], edges[..., 1]
    K = torch.zeros(Bn, n_nodes, n_nodes, dtype=G.dtype)
    for (i, j), sign in (((u, u), 1), ((v, v), 1), ((u, v), -1), ((v, u), -1)):
        K = K.index_put((b, i, j), sign * G, accumulate=True)
    return K


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/cow_workflow_dag/tensors.py
"""Build the concrete tensors: the bundled Circle of Willis crops, the constants, and the Snakemake references.

data/cases/<case>.npz     TopCoW MRA_Lausanne (CC0) crops around the Circle of Willis, 0.5 mm isotropic, 96×80×48
                          voxels, axes i → patient left, j → posterior, k → superior: `image` uint8 = 170 × the
                          normalised TOF-MRA intensity (I − p50)/(p99 − p50) clipped to [0, 1.5], `labels` uint8 =
                          TopCoW vessel classes (see LABELS). data/SOURCES.md has the processing steps.
data/Snakefile            the workflow: the same rules, run offline by Snakemake to make the references below
data/snakemake/           what Snakemake printed and measured (test references, never re-run here)
"""

import csv
import re
from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data"
SNAKEFILE = DATA / "Snakefile"
REF = DATA / "snakemake"

H = 0.5e-3  # voxel size (m)
MU = 1060.0 * 3.5e-6  # blood dynamic viscosity (Pa s), Newtonian: ρν with ρ = 1060 kg/m³, ν = 3.5e-6 m²/s
P_IN = 12.0e3  # mean arterial pressure at the inlets (Pa), about 90 mmHg; outlets drain to 0 through Murray loads
Q_TOT = 10e-6  # total cerebral inflow the outlet loads are tuned for (m³/s): ICAs ~4 ml/s each + basilar ~2 ml/s
MMHG = 133.322
THETA, WIDTH = 0.30, 0.02  # segmentation threshold on the normalised intensity, and the soft-threshold width (a.u.)

# TopCoW 2024 multiclass labels
LABELS = {1: "BA", 2: "R-PCA", 3: "L-PCA", 4: "R-ICA", 5: "R-MCA", 6: "L-ICA", 7: "L-MCA", 8: "R-Pcom", 9: "L-Pcom",
          10: "Acom", 11: "R-ACA", 12: "L-ACA", 15: "3rd-A2"}  # fmt: skip
INLETS = (4, 6, 1)  # ICA ends and the basilar end feed the circle (as in Venova: Pcom/Acom ends are no boundaries)
OUTLETS = (5, 7, 11, 12, 2, 3, 15)  # MCA, ACA, PCA (and a third A2) ends drain it
TERMINAL = INLETS + OUTLETS  # labels with a free end in the crop; 8, 9, 10 are connectors between two junctions
CONNECTIONS = (
    (1, 2),
    (1, 3),
    (2, 8),
    (3, 9),
    (4, 8),
    (6, 9),
    (4, 5),
    (4, 11),
    (5, 11),
    (6, 7),
    (6, 12),
    (7, 12),
    (10, 11),
    (10, 12),
    (10, 15),
    (11, 15),
    (12, 15),
)  # label pairs that join in the Circle of Willis
PAIRS = {"ICA": (4, 6), "MCA": (5, 7), "ACA": (11, 12), "PCA": (2, 3), "Pcom": (8, 9)}  # (right, left)
L_ICA = 6

CASES = ("sub-000", "sub-002", "sub-006", "sub-021", "sub-033", "sub-354")
STENOSIS_LENGTH = 10e-3  # length of the left-ICA stenosis in the severity sweep (m)
SWEEP_STEPS = 11  # severity 0, 0.1, …, 1 (fraction of the left-ICA lumen diameter lost)
CONDITIONS = ("baseline", "lica_occluded")  # both inlets open, or the left ICA inlet closed (a collateral test)


def load_case_file(path) -> tuple[torch.Tensor, torch.Tensor]:
    """One case file → image (1, 96, 80, 48) float64 (normalised intensity, a.u.) and labels (1, 96, 80, 48) int64."""
    with np.load(path) as d:
        image = torch.from_numpy(d["image"].astype(np.float64) / 170)
        labels = torch.from_numpy(d["labels"].astype(np.int64))
    return image[None], labels[None]


def load_cases(cases=CASES) -> tuple[torch.Tensor, torch.Tensor]:
    """Images and labels of several cases stacked on the batch axis: (B, 96, 80, 48) each."""
    pairs = [load_case_file(DATA / "cases" / f"{c}.npz") for c in cases]
    return torch.cat([p[0] for p in pairs]), torch.cat([p[1] for p in pairs])


# ---------------------------------------------------------------- Snakemake references (made offline, in WSL)
def ref_plan(name: str = "dryrun.txt") -> dict:
    """Jobs of a `snakemake -n` listing, by Snakemake job id: (rule, case, condition), inputs and outputs."""
    out = {}
    for block in re.split(r"\n(?=(?:local)?rule \w+:)", (REF / name).read_text(encoding="utf-8")):
        m = re.match(r"(?:local)?rule (\w+):", block.strip())
        if not m:
            continue
        field = {k: v.strip() for k, v in re.findall(r"^\s+(\w+): (.*)$", block, flags=re.M)}
        wc = dict(re.findall(r"(\w+)=([\w-]+)", field.get("wildcards", "")))
        job = (m.group(1), wc.get("case", ""), wc.get("condition", ""))
        ins, outs = ([f for f in field.get(k, "").split(", ") if f] for k in ("input", "output"))
        out[int(field["jobid"])] = {"job": job, "input": ins, "output": outs}
    return out


def ref_dag() -> tuple[set, set, set]:
    """Snakemake's job DAG: its jobs, the edges of `snakemake --dag` (dot, node ids = the dry run's job ids) and the
    edges implied by the dry run's input/output files, both as (consumer, input job)."""
    plan = ref_plan()
    text = (REF / "dag.dot").read_text(encoding="utf-8")
    for k, label in re.findall(r'^\s*(\d+)\[label = "([^"\\]*)', text, flags=re.M):
        assert plan[int(k)]["job"][0] == label  # the same job numbering
    dot = {(plan[int(b)]["job"], plan[int(a)]["job"]) for a, b in re.findall(r"^\s*(\d+) -> (\d+)", text, flags=re.M)}
    producer = {f: v["job"] for v in plan.values() for f in v["output"]}
    files = {(v["job"], producer[f]) for v in plan.values() for f in v["input"] if f in producer}
    return {v["job"] for v in plan.values()}, dot, files


def ref_jobs(name: str) -> list[tuple]:
    """Jobs a Snakemake run or dry run listed (csv: rule, case, condition), in its order."""
    with open(REF / name, encoding="utf-8") as f:
        rows = [r for r in csv.reader(line for line in f if not line.startswith("#"))]
    return [tuple(r) for r in rows[1:]]


def ref_timings(cores: int) -> list[dict]:
    """Per-job start/end (s, from the job's shell) of the Snakemake run with this many cores."""
    with open(REF / f"timings_cores{cores}.csv", encoding="utf-8") as f:
        rows = list(csv.DictReader(line for line in f if not line.startswith("#")))
    return [{**r, "start": float(r["start"]), "end": float(r["end"])} for r in rows]


def ref_runs() -> dict:
    """Whole-run wall-clock times of the Snakemake runs (s), keyed by cores, and the machine notes."""
    out = {}
    for line in (REF / "runs.csv").read_text(encoding="utf-8").splitlines():
        if line and not line.startswith(("#", "cores")):
            c, wall = line.split(",")[:2]
            out[int(c)] = float(wall)
    return out


def ref_cohort() -> list[dict]:
    """results/cohort.csv as Snakemake's jobs wrote it."""
    with open(REF / "cohort.csv", encoding="utf-8") as f:
        return list(csv.DictReader(f))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/cow_workflow_dag/solve.py
"""The pipeline stages, each a batched tensor function over the case axis (B cases at once, or B = 1 when Snakemake
runs one job), and the tensor workflow engine that reads the Snakefile and runs the whole DAG.

Stages:  segment (soft threshold + connected components) → centreline (min-plus eikonal on the labelled voxel graph)
         → measure (radii from the soft segmentation, lengths, asymmetry, stenosis) → boundaries (ICA/BA inlets,
         MCA/ACA/PCA outlets) → flow (Poiseuille network + Murray loads, one batched linear solve) → report → cohort.
Only torch is imported here, so the same file runs inside each Snakemake job.
"""

import math
import re
import time

import torch
import torch.nn.functional as F

from . import equation as eq
from . import tensors as T

MIN_CC = 64  # connected components smaller than this (voxels, 8 mm³) are noise
CLUSTER = 3.5e-3  # label contacts closer than this (m) are one junction
OFFSETS = [(a, b, c) for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1) if (a, b, c) > (0, 0, 0)]  # 13


# ---------------------------------------------------------------- voxel-graph helpers
def dilate(mask: torch.Tensor, r: int) -> torch.Tensor:
    """Cube dilation by r voxels, (B, X, Y, Z) bool."""
    return F.max_pool3d(mask.double()[:, None], 2 * r + 1, 1, r)[:, 0] > 0


def distance_map(mask: torch.Tensor, max_r: int = 16) -> torch.Tensor:
    """Depth of every voxel in the mask (m): erosions survived, alternating cube and cross elements (an octagonal
    approximation of the Euclidean distance to the wall); a voxel on the wall gets h/2."""
    x = mask.double()[:, None]
    d = torch.zeros_like(x)
    for k in range(max_r):
        d = d + x
        if k % 2 == 0:
            x = -F.max_pool3d(-x, 3, 1, 1)  # cube erosion (outside the crop counts as inside: no erosion there)
        else:
            xp = F.pad(x, (1, 1, 1, 1, 1, 1), mode="replicate")
            X, Y, Z = x.shape[2:]
            y = x
            for a, b, c in ((0, 1, 1), (2, 1, 1), (1, 0, 1), (1, 2, 1), (1, 1, 0), (1, 1, 2)):  # cross erosion
                y = torch.minimum(y, xp[:, :, a : a + X, b : b + Y, c : c + Z])
            x = y
        if not bool(x.any()):
            break
    return ((d - 0.5).clamp(min=0) * T.H)[:, 0]


def voxel_graph(mask: torch.Tensor):
    """Voxels of a (B, X, Y, Z) mask as nodes (coords (V, 4): case, i, j, k, ordered case by case) and their
    26-neighbour pairs (src, dst) once each, with the edge lengths (m)."""
    coords = mask.nonzero()
    index = torch.full(mask.shape, -1, dtype=torch.long)
    index[mask] = torch.arange(len(coords))
    shape = torch.tensor(mask.shape[1:])
    src, dst, ln = [], [], []
    for o in OFFSETS:
        n = coords[:, 1:] + torch.tensor(o)
        ok = ((n >= 0) & (n < shape)).all(1)
        nb = torch.full((len(coords),), -1, dtype=torch.long)
        nb[ok] = index[coords[ok, 0], n[ok, 0], n[ok, 1], n[ok, 2]]
        keep = nb >= 0
        src.append(torch.arange(len(coords))[keep]), dst.append(nb[keep])
        ln.append(torch.full((int(keep.sum()),), math.sqrt(sum(c * c for c in o)) * T.H, dtype=torch.float64))
    return coords, torch.cat(src), torch.cat(dst), torch.cat(ln)


def components(n: int, src: torch.Tensor, dst: torch.Tensor) -> torch.Tensor:
    """Connected components: every node takes the smallest node index of its component (min-label propagation with
    pointer jumping), (n,) long."""
    lab = torch.arange(n)
    while True:
        m = torch.minimum(lab[src], lab[dst])
        new = lab.scatter_reduce(0, src, m, "amin").scatter_reduce(0, dst, m, "amin")
        new = new[new]
        if torch.equal(new, lab):
            return lab
        lab = new


def eikonal(n: int, src: torch.Tensor, dst: torch.Tensor, w: torch.Tensor, sources: torch.Tensor):
    """Arrival times from the source nodes over undirected edges by min-plus sweeps, and each node's predecessor
    on a shortest path (itself for sources and unreached nodes)."""
    s2, d2, w2 = torch.cat([src, dst]), torch.cat([dst, src]), torch.cat([w, w])
    Tt = torch.full((n,), math.inf, dtype=torch.float64)
    Tt[sources] = 0.0
    while True:
        new = eq.relax(Tt, s2, d2, w2)
        if torch.equal(new, Tt):
            break
        Tt = new
    tight = (Tt[s2] + w2 == Tt[d2]) & torch.isfinite(Tt[d2])
    pred = torch.arange(n).scatter_reduce(0, d2[tight], s2[tight], "amin", include_self=False)
    pred[sources] = sources
    return Tt, pred


def per_case_rank(b: torch.Tensor, B: int) -> torch.Tensor:
    """Position of each item within its case, for items sorted by case: (n,) long."""
    counts = torch.bincount(b, minlength=B)
    start = torch.cat([torch.zeros(1, dtype=torch.long), counts.cumsum(0)[:-1]])
    return torch.arange(len(b)) - start[b]


def segment_argmin(values: torch.Tensor, group: torch.Tensor, n_groups: int) -> torch.Tensor:
    """For each group, the index of its smallest value (ties: the first index); −1 for an empty group."""
    best = torch.full((n_groups,), math.inf, dtype=values.dtype).scatter_reduce(0, group, values, "amin")
    hit = values == best[group]
    idx = torch.full((n_groups,), len(values), dtype=torch.long)
    idx = idx.scatter_reduce(0, group[hit], torch.arange(len(values))[hit], "amin")
    return torch.where(idx == len(values), -1, idx)


# ---------------------------------------------------------------- stage 1: segment
def segment(image: torch.Tensor, labels: torch.Tensor, theta: torch.Tensor, width: float = T.WIDTH, keep=None):
    """Threshold segmentation of the MRA: soft s = σ((I − θ)/w) (differentiable in θ), the hard mask I > θ minus
    components under MIN_CC voxels, its Dice against the TopCoW mask, and the region within 2 voxels of the labelled
    vessels (where the vessel calibre is measured). `keep` (the component mask) can be held fixed."""
    soft = eq.soft_threshold(image, theta, width)
    hard = image > theta.detach()[:, None, None, None]
    if keep is None:
        coords, src, dst, _ = voxel_graph(hard)
        lab = components(len(coords), src, dst)
        size = torch.bincount(lab, minlength=len(coords))[lab]
        keep = torch.zeros_like(hard)
        keep[hard] = size >= MIN_CC
        keep = dilate(keep, 1)  # room for the soft rim around each kept component
    truth = labels > 0
    mask = hard & keep
    return {"soft": soft * keep, "mask": mask, "keep": keep, "region": dilate(truth, 2), "dice": eq.dice(mask, truth)}


# ---------------------------------------------------------------- stage 2: centreline graph
def centreline(labels: torch.Tensor) -> dict:
    """Centreline graph of each case from the TopCoW labels. Junction nodes where labelled vessels touch (contacts
    closer than CLUSTER merged), a free-end node per terminal vessel (its voxel farthest from its junctions, by a
    min-plus eikonal with travel time ℓ/R²), and one edge per vessel piece between consecutive nodes along the
    shortest path. All vessels of all cases form one disjoint voxel graph, so each eikonal is one batched loop."""
    B = labels.shape[0]
    vessel = labels > 0
    R = distance_map(vessel)
    coords, src, dst, ln = voxel_graph(vessel)
    V = len(coords)
    bv, lv = coords[:, 0], labels[vessel]
    pos = coords[:, 1:].double() * T.H  # (V, 3) voxel centres (m)
    Rv = R[vessel]

    # largest connected piece of each labelled vessel (an instance = one (case, label))
    same = lv[src] == lv[dst]
    comp = components(V, src[same], dst[same])
    size = torch.bincount(comp, minlength=V)
    key = bv * 16 + lv
    score = size[comp] * (V + 1) + comp
    best = torch.zeros(16 * B, dtype=torch.long).scatter_reduce(0, key, score, "amax")
    kept = score == best[key]
    ekeep = kept[src] & kept[dst]

    # contacts between different labels → junction clusters
    cross = ekeep & ~same
    cs, cd = src[cross], dst[cross]
    la, lb = torch.minimum(lv[cs], lv[cd]), torch.maximum(lv[cs], lv[cd])
    # only anatomical connections count: e.g. two A2s running side by side touch but do not communicate; the two
    # ACAs count as connected only in a case without an Acom label (a fused or very short Acom)
    pair = la * 16 + lb
    allowed = torch.isin(pair, torch.tensor([a * 16 + b for a, b in T.CONNECTIONS]))
    no_acom = ~torch.zeros(B, dtype=torch.bool).index_fill(0, bv[lv == 10].unique(), True)
    allowed |= (pair == 11 * 16 + 12) & no_acom[bv[cs]]
    cs, cd, la, lb = cs[allowed], cd[allowed], la[allowed], lb[allowed]
    ckey = bv[cs] * 256 + la * 16 + lb
    ukey, cinv = torch.unique(ckey, return_inverse=True)
    C = len(ukey)
    cnt = torch.bincount(cinv, minlength=C).double()
    cpos = torch.zeros(C, 3, dtype=torch.float64).index_add(0, cinv, (pos[cs] + pos[cd]) / 2) / cnt[:, None]
    cb, cla, clb = ukey // 256, (ukey // 16) % 16, ukey % 16
    near = (torch.cdist(cpos, cpos) < CLUSTER) & (cb[:, None] == cb[None, :])
    reach = eq.closure(near.double()) | torch.eye(C, dtype=torch.bool)  # transitive closure: one cluster each
    rep = reach.double().argmax(1)  # smallest contact index in the cluster
    jroots, cnode = torch.unique(rep, return_inverse=True)  # junction nodes (sorted by case, then first contact)
    NJ = len(jroots)
    jb = cb[jroots]
    jpos = torch.zeros(NJ, 3, dtype=torch.float64).index_add(0, cnode, cpos)
    jpos = jpos / torch.bincount(cnode, minlength=NJ).double()[:, None]

    # instances and their attached junctions
    inst_keys, vinst = torch.unique(key[kept], return_inverse=True)
    K = len(inst_keys)
    inst_of_key = torch.full((16 * B,), -1, dtype=torch.long)
    inst_of_key[inst_keys] = torch.arange(K)
    ib, il = inst_keys // 16, inst_keys % 16
    att = torch.unique(torch.cat([torch.stack([inst_of_key[cb * 16 + cla], cnode], 1),
                                  torch.stack([inst_of_key[cb * 16 + clb], cnode], 1)]), dim=0)  # fmt: skip
    att = att[att[:, 0] >= 0]
    a_inst, a_node = att[:, 0], att[:, 1]
    n_att = torch.bincount(a_inst, minlength=K)
    terminal = torch.isin(il, torch.tensor(T.TERMINAL))
    usable = torch.where(terminal, n_att >= 1, n_att >= 2)

    # the kept voxel graph (re-indexed), with eikonal weights ℓ/R²
    kidx = torch.full((V,), -1, dtype=torch.long)
    kidx[kept] = torch.arange(int(kept.sum()))
    ks, kd = kidx[src[ekeep & same]], kidx[dst[ekeep & same]]
    kpos, kR = pos[kept], Rv[kept]
    w = eq.edge_cost(ln[ekeep & same], (kR[ks] + kR[kd]) / 2)
    nk = len(kpos)

    # seed voxel of each attachment: the instance voxel nearest to the junction
    vox_by_inst = torch.argsort(vinst, stable=True)
    first = torch.cat([torch.zeros(1, dtype=torch.long), torch.bincount(vinst, minlength=K).cumsum(0)])
    n_in = first[a_inst + 1] - first[a_inst]
    rep_a = torch.repeat_interleave(torch.arange(len(att)), n_in)
    within = torch.arange(int(n_in.sum())) - torch.repeat_interleave(torch.cumsum(n_in, 0) - n_in, n_in)
    cand = vox_by_inst[first[a_inst][rep_a] + within]
    dist = (kpos[cand] - jpos[a_node][rep_a]).norm(dim=1)
    a_seed = cand[segment_argmin(dist, rep_a, len(att))]

    # eikonal 1 from all junction seeds: each terminal vessel's free end is its farthest voxel along the vessel
    T1, _ = eikonal(nk, ks, kd, ln[ekeep & same], a_seed)  # plain path length: thin side branches lose
    T1i = torch.where(torch.isfinite(T1), T1, torch.tensor(-1.0, dtype=torch.float64))
    far = torch.full((K,), -1.0, dtype=torch.float64).scatter_reduce(0, vinst, T1i, "amax")
    hit = T1i == far[vinst]
    far_vox = torch.full((K,), nk, dtype=torch.long).scatter_reduce(0, vinst[hit], torch.arange(nk)[hit], "amin")
    # re-centre the end: the most central voxel (largest R) within 2.5 mm of the farthest one (a cap or a crop face)
    dfar = (kpos - kpos[far_vox.clamp(max=nk - 1)][vinst]).norm(dim=1)
    score = torch.where(dfar < 2.5e-3, kR - 1e-3 * dfar, torch.tensor(-math.inf, dtype=torch.float64))
    end_vox = segment_argmin(-score, vinst, K)
    # the first attachment of each instance (its connector start)
    first_att = torch.full((K,), len(att), dtype=torch.long).scatter_reduce(0, a_inst, torch.arange(len(att)), "amin")
    start_vox = torch.where(terminal, end_vox, a_seed[first_att.clamp(max=len(att) - 1)])
    use_k = usable.nonzero().flatten()

    # eikonal 2 from each instance's start: orders its junctions and gives the path back to the start
    T2, pred = eikonal(nk, ks, kd, w, start_vox[use_k])
    a_use = usable[a_inst] & ~(~terminal[a_inst] & (torch.arange(len(att)) == first_att[a_inst]))
    au = a_use.nonzero().flatten()  # attachments that become chain points after the start
    order = torch.argsort(a_inst[au] * 1e6 + T2[a_seed[au]] / T2[torch.isfinite(T2)].max().clamp(min=1e-30))
    au = au[order]  # by instance, then by distance from the start
    lastpos = torch.full((K,), -1, dtype=torch.long).scatter_reduce(0, a_inst[au], torch.arange(len(au)), "amax")
    last = torch.where(lastpos >= 0, au[lastpos.clamp(min=0)], -1)  # each instance's farthest junction
    # backtrack all instances at once from their farthest junction seed
    cur = a_seed[last[use_k]]
    paths = [cur]
    while True:
        nxt = pred[cur]
        if torch.equal(nxt, cur):
            break
        cur = nxt
        paths.append(cur)
    P = torch.stack(paths[::-1], 1)  # (Ku, Lmax) start … farthest seed, left-padded with the start voxel repeated
    pad_n = (P == P[:, :1]).sum(1) - 1  # leading repeats to drop
    Lmax = P.shape[1]
    shift = (torch.arange(Lmax)[None, :] + pad_n[:, None]).clamp(max=Lmax - 1)
    P = P.gather(1, shift)
    plen = Lmax - pad_n  # valid points per path
    pmask = torch.arange(Lmax)[None, :] < plen[:, None]

    # chain points: start, then each attachment's nearest path index
    ku_of = torch.full((K,), -1, dtype=torch.long)
    ku_of[use_k] = torch.arange(len(use_k))
    row = ku_of[a_inst[au]]
    d_pa = (kpos[P[row]] - kpos[a_seed[au]][:, None, :]).norm(dim=2)
    d_pa = torch.where(pmask[row], d_pa, torch.tensor(math.inf, dtype=torch.float64))
    cut = d_pa.argmin(1)
    cut = torch.where(au == last[a_inst[au]], plen[row] - 1, cut)  # the farthest one ends the path
    big = Lmax + 1  # cuts never go back along a path: running maximum within each instance
    cut = torch.cummax(a_inst[au] * big + cut, 0).values - a_inst[au] * big
    # node ids per case: junctions first, then one free-end node per usable terminal instance
    j_rank = per_case_rank(jb, B)
    nj_case = torch.bincount(jb, minlength=B)
    te = use_k[terminal[use_k]]
    e_rank = per_case_rank(ib[te], B)
    end_node_local = torch.full((K,), -1, dtype=torch.long)
    end_node_local[te] = nj_case[ib[te]] + e_rank
    start_node = torch.where(terminal, end_node_local, j_rank[a_node[first_att.clamp(max=len(att) - 1)]])
    # edges: (start → first attachment), (attachment → next attachment), … per instance
    prev_same = torch.cat([torch.tensor([False]), a_inst[au][1:] == a_inst[au][:-1]])
    u_node = torch.where(prev_same, torch.roll(j_rank[a_node[au]], 1), start_node[a_inst[au]])
    u_cut = torch.where(prev_same, torch.roll(cut, 1), torch.zeros_like(cut))
    v_node, v_cut = j_rank[a_node[au]], cut
    e_case, e_label = ib[a_inst[au]], il[a_inst[au]]

    # polylines between the cuts: path voxel centres, ends pinned to the node positions, lightly smoothed
    nseg = (v_cut - u_cut + 1).clamp(min=2)
    Pm = int(nseg.max())
    idx = (u_cut[:, None] + torch.arange(Pm)[None, :]).clamp(max=Lmax - 1)
    ok = torch.arange(Pm)[None, :] < nseg[:, None]
    idx = torch.minimum(idx, v_cut[:, None])
    poly = kpos[P[row].gather(1, idx)]  # (E, Pm, 3)
    n_total = nj_case + torch.bincount(ib[te], minlength=B)
    N = int(n_total.max())
    node_pos = torch.full((B, N, 3), math.nan, dtype=torch.float64)
    node_pos[jb, j_rank] = jpos
    node_pos[ib[te], end_node_local[te]] = kpos[end_vox[te]]
    node_kind = torch.full((B, N), -1, dtype=torch.long)
    node_kind[jb, j_rank] = 0
    node_kind[ib[te], end_node_local[te]] = il[te]
    poly[:, 0] = node_pos[e_case, u_node]
    last_i = (nseg - 1)[:, None, None].expand(-1, 1, 3)
    poly = poly.scatter(1, last_i, node_pos[e_case, v_node][:, None, :])
    poly = smooth_polyline(poly, ok)
    seg_len = torch.where(ok[:, 1:], (poly[:, 1:] - poly[:, :-1]).norm(dim=2), torch.zeros(1, dtype=torch.float64))
    radius_dt = torch.where(ok, kR[P[row].gather(1, idx)], torch.zeros(1, dtype=torch.float64)).sum(1) / nseg

    # pack per case: (B, E, …), padded
    er = per_case_rank(e_case, B)
    E = int(torch.bincount(e_case, minlength=B).max())
    edges = torch.zeros(B, E, 2, dtype=torch.long)
    edges[e_case, er] = torch.stack([u_node, v_node], 1)
    out_poly = torch.full((B, E, Pm, 3), math.nan, dtype=torch.float64)
    out_poly[e_case, er] = torch.where(ok[..., None], poly, torch.tensor(math.nan, dtype=torch.float64))
    pack = lambda x, fill: torch.full((B, E), fill, dtype=x.dtype).index_put((e_case, er), x)  # noqa: E731
    return {
        "node_pos": node_pos, "node_kind": node_kind, "node_valid": node_kind >= 0,
        "edges": edges, "edge_label": pack(e_label, 0),
        "edge_valid": pack(torch.ones_like(e_label, dtype=torch.bool), False),
        "poly": out_poly, "length": pack(seg_len.sum(1), 0.0), "radius_dt": pack(radius_dt, 0.0),
    }  # fmt: skip


def smooth_polyline(poly: torch.Tensor, ok: torch.Tensor, half: int = 2) -> torch.Tensor:
    """Moving average over ±half points along each polyline (E, P, 3), end points kept."""
    x = torch.where(ok[..., None], poly, torch.zeros(1, dtype=poly.dtype)).transpose(1, 2)  # (E, 3, P)
    k = torch.ones(1, 1, 2 * half + 1, dtype=poly.dtype)
    num = F.conv1d(x.reshape(-1, 1, x.shape[-1]), k, padding=half).reshape(x.shape)
    den = F.conv1d(ok.double()[:, None], k, padding=half)
    sm = (num / den.clamp(min=1)).transpose(1, 2)
    n = ok.sum(1)
    pin = (torch.arange(poly.shape[1])[None, :] == 0) | (torch.arange(poly.shape[1])[None, :] == (n - 1)[:, None])
    return torch.where(pin[..., None] | ~ok[..., None], poly, sm)


# ---------------------------------------------------------------- stage 3: measure
def measure(seg: dict, g: dict, chunk: int = 2048, near=None) -> dict:
    """Calibre of every edge from the soft segmentation: each voxel of the vessel region goes to its nearest
    centreline point, the soft volume per edge V_e = Σ s h³ gives r_e = √(V_e/(πL_e)), differentiable in θ; the same
    per point gives a radius profile (stenosis: a dip below half the median radius). Also L/R asymmetry per vessel."""
    soft, region = seg["soft"], seg["region"]
    B, E, Pm, _ = g["poly"].shape
    flat = g["poly"].reshape(B, E * Pm, 3)
    valid = ~torch.isnan(flat[..., 0])
    pb, pq = valid.nonzero(as_tuple=True)  # the real centreline points, packed per case (order kept)
    pr = per_case_rank(pb, B)
    pts = torch.full((B, int(torch.bincount(pb, minlength=B).max()), 3), 1e3, dtype=torch.float64)  # padding far away
    pts[pb, pr] = flat[pb, pq]
    pidx = torch.zeros(pts.shape[:2], dtype=torch.long).index_put((pb, pr), pq)
    coords = region.nonzero()
    rb = coords[:, 0]
    rr = per_case_rank(rb, B)
    Vr = int(torch.bincount(rb, minlength=B).max())
    vox = torch.full((B, Vr, 3), -1e3, dtype=torch.float64)
    vox[rb, rr] = coords[:, 1:].double() * T.H
    if near is None:  # the voxel → centreline-point assignment does not depend on θ: it can be reused
        exact = "donot_use_mm_for_euclid_dist"  # exact distances: the same nearest point for B = 1 and B = 6
        near = torch.cat([torch.cdist(vox[:, i : i + chunk], pts, compute_mode=exact).argmin(2)
                          for i in range(0, Vr, chunk)], 1)  # fmt: skip
        near = pidx.gather(1, near)  # (B, Vr) index into the E·Pm points
    s = torch.zeros(B, Vr, dtype=soft.dtype).index_put((rb, rr), soft[region])  # soft occupancy per region voxel
    vol_pt = torch.zeros(B, E * Pm, dtype=soft.dtype).scatter_add(1, near, s * T.H**3)  # (B, E·Pm)
    vol = vol_pt.reshape(B, E, Pm).sum(2)
    L = g["length"]
    r = torch.where(g["edge_valid"], (vol / (math.pi * L.clamp(min=1e-9))).clamp(min=0).sqrt(), torch.zeros_like(vol))
    # radius profile and stenosis (detached: a report flag)
    okp = valid.reshape(B, E, Pm)
    ds = (L / (okp.sum(2) - 1).clamp(min=1)).detach()
    rp = (vol_pt.detach().reshape(B, E, Pm) / (math.pi * ds[..., None].clamp(min=1e-9))).sqrt()
    k = torch.ones(1, 1, 9, dtype=torch.float64)
    num = F.conv1d(torch.where(okp, rp, 0.0).reshape(-1, 1, Pm), k, padding=4).reshape(B, E, Pm)
    den = F.conv1d(okp.double().reshape(-1, 1, Pm), k, padding=4).reshape(B, E, Pm)
    rs = num / den.clamp(min=1)
    n = okp.sum(2, keepdim=True)
    i = torch.arange(Pm)[None, None, :]
    inner = okp & (i >= 4) & (i < n - 4)
    rmin = torch.where(inner, rs, math.inf).min(2).values
    rmed = torch.where(inner, rs, math.nan).nanmedian(2).values
    severity = torch.where(inner.sum(2) >= 4, 1 - rmin / rmed, torch.zeros_like(rmin)).nan_to_num(0.0)
    # mean radius per vessel label (length-weighted) and the right/left asymmetry 2(r_R − r_L)/(r_R + r_L)
    onehot = F.one_hot(g["edge_label"], 16).double() * g["edge_valid"][..., None]
    rlab = torch.einsum("bel,be->bl", onehot, r * L) / torch.einsum("bel,be->bl", onehot, L).clamp(min=1e-12)
    present = torch.einsum("bel->bl", onehot) > 0
    asym = {}
    for name, (right, left) in T.PAIRS.items():
        both = present[:, right] & present[:, left]
        ai = 2 * (rlab[:, right] - rlab[:, left]) / (rlab[:, right] + rlab[:, left]).clamp(min=1e-12)
        asym[name] = torch.where(both, ai, torch.tensor(math.nan, dtype=torch.float64))
    return {"near": near, "dice": seg["dice"], "radius": r, "volume": vol, "length": L, "profile": rs,
            "severity": severity, "stenosis": severity >= 0.5,
            "r_label": rlab, "present": present, "asym": asym}  # fmt: skip


# ---------------------------------------------------------------- stage 4: boundaries
def boundaries(g: dict) -> dict:
    """Inlets at the free ends of the ICAs and the basilar artery, outlets at the free ends of the MCAs, ACAs and
    PCAs (Pcom and Acom ends are never boundaries). Each boundary gets its terminal edge and an outward normal."""
    kind = g["node_kind"]
    inlet = torch.isin(kind, torch.tensor(T.INLETS))
    outlet = torch.isin(kind, torch.tensor(T.OUTLETS))
    B, E, Pm, _ = g["poly"].shape
    u = g["edges"][..., 0]
    is_end = (kind.gather(1, u) > 0) & g["edge_valid"]  # edges that start at a free end (the first of a chain)
    bedge = torch.full(kind.shape, -1, dtype=torch.long)
    b_idx, e_idx = is_end.nonzero(as_tuple=True)
    bedge[b_idx, u[b_idx, e_idx]] = e_idx
    n_pts = (~torch.isnan(g["poly"][..., 0])).sum(2)
    k_in = (n_pts.clamp(min=2) - 1).clamp(max=4)  # a point ~2 mm into the vessel
    inner = g["poly"].gather(2, k_in[..., None, None].expand(B, E, 1, 3))[:, :, 0]
    direction = g["poly"][:, :, 0] - inner
    direction = direction / direction.norm(dim=2, keepdim=True).clamp(min=1e-12)
    normal = torch.zeros(B, kind.shape[1], 3, dtype=torch.float64)
    normal[b_idx, u[b_idx, e_idx]] = direction[b_idx, e_idx]
    return {"inlet": inlet, "outlet": outlet, "edge": bedge, "normal": normal}


# ---------------------------------------------------------------- stage 5: flow
def flow(m: dict, bnd: dict, g: dict, occluded: torch.Tensor, severity: torch.Tensor | None = None) -> dict:
    """Steady Poiseuille flow in the vessel network: conductance G_e per edge, Murray loads g_k to 0 Pa at the
    outlets, P_in at every open inlet (the left ICA inlet closed where `occluded`), one batched solve K p = b.

    `severity` (B,) in [0, 1] generalises `occluded`: the left-ICA inlet is fed through a stenosis of length
    STENOSIS_LENGTH whose lumen diameter is (1 − s) of the inlet vessel's, in series (Poiseuille: conductance
    G_s = G_ref a/(1 − a), a = (1 − s)⁴, so the inlet's total conductance drops as a). s = 0 is the open Dirichlet
    inlet and s = 1 the closed one, exactly the two workflow conditions."""
    G = torch.where(g["edge_valid"], eq.poiseuille_conductance(m["radius"], m["length"], T.MU), 0.0)
    B, N = g["node_kind"].shape
    r_out = m["radius"].gather(1, bnd["edge"].clamp(min=0)) * (bnd["edge"] >= 0)
    g_out = eq.murray_conductance(r_out, bnd["outlet"] & (bnd["edge"] >= 0), T.Q_TOT, T.P_IN)
    s = (occluded.to(G.dtype) if severity is None else severity.to(G.dtype))[:, None]
    lica = bnd["inlet"] & (g["node_kind"] == T.L_ICA)
    inlet = bnd["inlet"] & ~(lica & (s > 0))  # Dirichlet inlets
    a = (1 - s).clamp(min=0) ** 4
    stenosed = lica & (s > 0) & (s < 1)
    r_in = m["radius"].gather(1, bnd["edge"].clamp(min=0))  # radius of the vessel at each free end
    g_ref = eq.poiseuille_conductance(r_in, torch.full_like(r_in, T.STENOSIS_LENGTH), T.MU)
    g_s = torch.where(stenosed, g_ref * a / (1 - a).clamp(min=1e-12), 0.0)  # inlet fed through the stenosis
    leak = 1e-9 * T.Q_TOT / T.P_IN  # a negligible leak to 0 Pa: keeps a node cut off from every boundary at p = 0
    K = eq.kirchhoff_matrix(G, g["edges"], N) + torch.diag_embed(g_out + leak + g_s)
    eye = torch.eye(N, dtype=K.dtype).expand(B, N, N)
    K = torch.where(inlet[..., None], eye, K)  # Dirichlet rows p = P_in
    p = torch.linalg.solve(K, torch.where(inlet, T.P_IN, g_s * T.P_IN).to(K.dtype))
    u, v = g["edges"][..., 0], g["edges"][..., 1]
    q = G * (p.gather(1, u) - p.gather(1, v))  # (B, E) flow u → v (m³/s)
    net_out = torch.zeros_like(p).scatter_add(1, u, q).scatter_add(1, v, -q)  # flow leaving each node via edges
    q_in = torch.where(inlet, net_out, torch.where(stenosed, g_s * (T.P_IN - p), 0.0))
    return {"p": p, "q": q, "Q_out": g_out * p, "Q_in": q_in, "G": G, "inlet": inlet | stenosed}


def stenosis_sweep(m: dict, bnd: dict, g: dict, steps: int = T.SWEEP_STEPS) -> tuple[torch.Tensor, list[dict]]:
    """Left-ICA severity s = 0 … 1 in `steps` steps: one batched flow solve over all cases per step."""
    sev = torch.linspace(0, 1, steps, dtype=torch.float64)
    B = g["node_kind"].shape[0]
    return sev, [flow(m, bnd, g, torch.zeros(B, dtype=torch.bool), torch.full((B,), float(s))) for s in sev]


# ---------------------------------------------------------------- stage 6: report, stage 7: cohort
COLUMNS = ["case", "condition", "dice", "nodes", "edges", "length_mm", "stenoses"] + [
    f"r_{n}_mm" for n in ("BA", "R-ICA", "L-ICA", "R-MCA", "L-MCA", "R-ACA", "L-ACA", "R-PCA", "L-PCA", "R-Pcom",
                         "L-Pcom", "Acom")] + [f"AI_{n}" for n in T.PAIRS] + [
    "Q_in_R-ICA", "Q_in_L-ICA", "Q_in_BA", "Q_R-MCA", "Q_L-MCA", "Q_R-ACA", "Q_L-ACA", "Q_R-PCA", "Q_L-PCA",
    "Q_out_total", "share_L-MCA", "dp_max_mmHg"]  # fmt: skip
NAME_TO_LABEL = {v: k for k, v in T.LABELS.items()}


def report(m: dict, bnd: dict, g: dict, fl: dict) -> dict:
    """The numbers of one (case, condition): (B,) tensors keyed by column (flows in ml/s, radii in mm)."""
    kind = g["node_kind"]
    out = {"dice": m["dice"], "nodes": g["node_valid"].sum(1).double(), "edges": g["edge_valid"].sum(1).double(),
           "length_mm": g["length"].sum(1) * 1e3,
           "stenoses": (m["stenosis"] & g["edge_valid"]).sum(1).double()}  # fmt: skip
    for n in ("BA", "R-ICA", "L-ICA", "R-MCA", "L-MCA", "R-ACA", "L-ACA", "R-PCA", "L-PCA", "R-Pcom", "L-Pcom", "Acom"):
        lab = NAME_TO_LABEL[n]
        out[f"r_{n}_mm"] = torch.where(m["present"][:, lab], m["r_label"][:, lab] * 1e3, math.nan)
    for n in T.PAIRS:
        out[f"AI_{n}"] = m["asym"][n]

    def at(values, label):  # sum of a node quantity over the free ends with this label
        return torch.where(kind == label, values, 0.0).sum(1) * 1e6

    for n in ("R-ICA", "L-ICA", "BA"):
        out[f"Q_in_{n}"] = at(fl["Q_in"], NAME_TO_LABEL[n])
    for n in ("R-MCA", "L-MCA", "R-ACA", "L-ACA", "R-PCA", "L-PCA"):
        out[f"Q_{n}"] = at(fl["Q_out"], NAME_TO_LABEL[n])
    out["Q_out_total"] = fl["Q_out"].sum(1) * 1e6
    out["share_L-MCA"] = out["Q_L-MCA"] / out["Q_out_total"].clamp(min=1e-12)
    p_out = torch.where(bnd["outlet"], fl["p"], math.inf).min(1).values
    out["dp_max_mmHg"] = (T.P_IN - p_out) / T.MMHG
    return out


def cohort_csv(rows: list[dict]) -> str:
    """The cohort table (one row per case and condition) as CSV text, numbers to 9 significant digits."""
    lines = [",".join(COLUMNS)]
    for r in rows:
        lines.append(",".join(r[c] if c in ("case", "condition") else f"{float(r[c]):.9g}" for c in COLUMNS))
    return "\n".join(lines) + "\n"


# ---------------------------------------------------------------- the tensor workflow engine
def parse_snakefile(text: str) -> list[dict]:
    """Rules of the Snakefile in order: name, output patterns, and inputs as (pattern, aggregated wildcards) where
    expand(pattern, case=…, …) aggregates over the named wildcards."""
    rules = []
    for block in re.split(r"^rule ", text, flags=re.M)[1:]:
        name = block.split(":", 1)[0].strip()
        sec = dict(re.findall(r"^    (input|output|log|shell):\s*(.*?)(?=^    \w+:|^\S|\Z)", block, flags=re.M | re.S))
        ins = []
        for m in re.finditer(r'expand\(\s*"([^"]+)"([^)]*)\)|"([^"]+)"', sec.get("input", "")):
            if m.group(1):
                ins.append((m.group(1), tuple(re.findall(r"(\w+)\s*=", m.group(2)))))
            else:
                ins.append((m.group(3), ()))
        outs = re.findall(r'"([^"]+)"', sec.get("output", ""))
        rules.append({"name": name, "inputs": ins, "outputs": outs})
    return rules


def rule_graph(rules: list[dict]) -> tuple[torch.Tensor, list[tuple]]:
    """A_rule (R, R): (A_rule)_{sr} = 1 when rule s reads a file pattern rule r writes; and each rule's wildcards
    (those of its outputs, or of its inputs for a target rule without outputs)."""
    R = len(rules)
    A = torch.zeros(R, R)
    for s, rs in enumerate(rules):
        for pat, _ in rs["inputs"]:
            for r, rr in enumerate(rules):
                if pat in rr["outputs"]:
                    A[s, r] = 1
    axes = []
    for r in rules:
        pats = r["outputs"] or [p for p, agg in r["inputs"]]
        wc = {w for p in pats for w in re.findall(r"\{(\w+)\}", p)}
        wc -= {w for p, agg in r["inputs"] for w in agg} if not r["outputs"] else set()
        axes.append(tuple(a for a in ("case", "condition") if a in wc))
    return A, axes


def job_list(rules: list[dict], axes: list, sizes: dict, values: dict) -> list[tuple]:
    """Jobs as (rule, case, condition), numbered rule by rule and row-major over the wildcard axes."""
    jobs = []
    for r, ax in zip(rules, axes):
        grids = [values[a] if a in ax else ("",) for a in sizes]
        for case in grids[0]:
            for cond in grids[1]:
                jobs.append((r["name"], case, cond))
    return jobs


def build_dag(text: str, cases=T.CASES, conditions=T.CONDITIONS) -> dict:
    """Everything the engine knows about the workflow: rules, A_rule, job list, job adjacency A (Kronecker), closure."""
    rules = parse_snakefile(text)
    A_rule, axes = rule_graph(rules)
    sizes = {"case": len(cases), "condition": len(conditions)}
    A, off = eq.job_adjacency(A_rule, axes, sizes)
    jobs = job_list(rules, axes, sizes, {"case": cases, "condition": conditions})
    return {"rules": rules, "A_rule": A_rule, "axes": axes, "A": A, "off": off, "jobs": jobs, "sizes": sizes}


def stage_functions(theta: torch.Tensor) -> dict:
    """What each rule does, as one batched call over all its jobs. `store` holds every finished rule's outputs.
    Case-axis stages see B cases; the (case, condition) stages see B·C rows, case-major like the job numbering."""

    def seg(st):
        return segment(st["image"], st["labels"], theta)

    def rep(x, C):  # broadcast case-level tensors to (case, condition) rows: the I ⊗ 𝟙 edge
        return {k: rep(v, C) if isinstance(v, dict) else v.repeat_interleave(C, 0) for k, v in x.items()}

    def flw(st):
        C = len(st["conditions"])
        occl = torch.tensor([c == "lica_occluded" for c in st["conditions"]] * len(st["cases"]))
        g, m, b = rep(st["centreline"], C), rep(st["measure"], C), rep(st["boundaries"], C)
        return {"flow": flow(m, b, g, occl), "g": g, "b": b}

    def rpt(st):
        C = len(st["conditions"])
        f = st["flow"]
        return report(rep(st["measure"], C), f["b"], f["g"], f["flow"])

    def coh(st):
        r = st["report"]
        rows = [{"case": c, "condition": k} for c in st["cases"] for k in st["conditions"]]
        for i, row in enumerate(rows):
            row.update({col: r[col][i] for col in r})
        return cohort_csv(rows)

    return {"segment": seg, "centreline": lambda st: centreline(st["labels"]),
            "measure": lambda st: measure(st["segment"], st["centreline"]),
            "boundaries": lambda st: boundaries(st["centreline"]), "flow": flw, "report": rpt, "cohort": coh,
            "all": lambda st: None}  # fmt: skip


def run_engine(dag: dict, cases=T.CASES, conditions=T.CONDITIONS, theta: float = T.THETA, grad: bool = False):
    """Run the DAG as tensors: rules level by level (levels of A_rule), each rule ONE batched call over its jobs.
    Returns the store and the wall-clock (start, end) of each rule call, relative to the start (s)."""
    t0 = time.perf_counter()
    image, labels = T.load_cases(cases)
    th = torch.full((len(cases),), theta, dtype=torch.float64, requires_grad=grad)
    store = {"image": image, "labels": labels, "cases": cases, "conditions": conditions, "theta": th}
    times = {"load": (0.0, time.perf_counter() - t0)}
    funcs = stage_functions(th)
    lev = eq.levels(dag["A_rule"])
    for k in range(int(lev.max()) + 1):
        for r in (lev == k).nonzero().flatten().tolist():
            name = dag["rules"][r]["name"]
            a = time.perf_counter() - t0
            with torch.set_grad_enabled(grad):
                store[name] = funcs[name](store)
            times[name] = (a, time.perf_counter() - t0)
    return store, times


# ---------------------------------------------------------------- the 3D scene: iso-surface of the segmentation
CUBE = torch.tensor([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0], [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1]])
TETS = torch.tensor([[0, 5, 1, 6], [0, 1, 2, 6], [0, 2, 3, 6], [0, 3, 7, 6], [0, 7, 4, 6], [0, 4, 5, 6]])


def iso_surface(field: torch.Tensor, level: float, h: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Marching tetrahedra (the table-free member of the marching-cubes family): every cube that straddles the level
    is cut into 6 tetrahedra; a tetrahedron with 1 or 3 corners inside gives a triangle, with 2 a quad (2 triangles).
    Vertices sit on the cut grid edges (linear interpolation) and are shared, so the surface is watertight.
    field (X, Y, Z), inside where field > level → vertices (V, 3) in m and triangles (F, 3), normals outward."""
    X, Y, Z = field.shape
    inside = field > level
    c = torch.stack(torch.meshgrid(torch.arange(X - 1), torch.arange(Y - 1), torch.arange(Z - 1), indexing="ij"), -1)
    c = c.reshape(-1, 3)
    corner = c[:, None, :] + CUBE[None]  # (C, 8, 3)
    flat = (corner[..., 0] * Y + corner[..., 1]) * Z + corner[..., 2]  # (C, 8) global grid index
    ins = inside.flatten()[flat]
    mixed = ins.any(1) & ~ins.all(1)
    tet = flat[mixed][:, TETS].reshape(-1, 4)  # (Nt, 4)
    tin = inside.flatten()[tet]
    n_in = tin.sum(1)
    tet, tin, n_in = tet[(n_in > 0) & (n_in < 4)], tin[(n_in > 0) & (n_in < 4)], n_in[(n_in > 0) & (n_in < 4)]
    order = torch.argsort((~tin).long(), dim=1, stable=True)  # inside corners first
    t = tet.gather(1, order)
    one = n_in == 1
    three = n_in == 3
    two = n_in == 2
    tri_edges = []  # each triangle as 3 grid edges (a, b), a inside, b outside
    a1 = t[one]
    tri_edges.append(torch.stack([torch.stack([a1[:, 0], a1[:, k]], 1) for k in (1, 2, 3)], 1))
    a3 = t[three]
    tri_edges.append(torch.stack([torch.stack([a3[:, k], a3[:, 3]], 1) for k in (0, 1, 2)], 1))
    a2 = t[two]
    ac, ad, bd, bc = (torch.stack([a2[:, i], a2[:, j]], 1) for i, j in ((0, 2), (0, 3), (1, 3), (1, 2)))
    tri_edges += [torch.stack([ac, ad, bd], 1), torch.stack([ac, bd, bc], 1)]
    te = torch.cat(tri_edges)  # (F, 3, 2)
    key = te.min(2).values * (X * Y * Z) + te.max(2).values
    ukey, inv = torch.unique(key.flatten(), return_inverse=True)
    ea, eb = ukey // (X * Y * Z), ukey % (X * Y * Z)
    f = field.flatten().double()
    grid = torch.stack(torch.meshgrid(torch.arange(X), torch.arange(Y), torch.arange(Z), indexing="ij"), -1)
    grid = grid.reshape(-1, 3).double()
    s = ((level - f[ea]) / (f[eb] - f[ea])).clamp(0, 1)[:, None]
    verts = (grid[ea] + s * (grid[eb] - grid[ea])) * h
    tris = inv.reshape(-1, 3)
    # orient outward: the normal must point from the inside corner to the outside one
    a, b, cc = verts[tris[:, 0]], verts[tris[:, 1]], verts[tris[:, 2]]
    nrm = torch.cross(b - a, cc - a, dim=1)
    out_dir = grid[te[:, 0, 1]] - grid[te[:, 0, 0]]
    flip = torch.einsum("fi,fi->f", nrm, out_dir) < 0
    tris = torch.where(flip[:, None], tris[:, [0, 2, 1]], tris)
    area = nrm.norm(dim=1)
    return verts, tris[area > 1e-30 * h * h]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/cow_workflow_dag/plot.py
"""The animation: six cases (MRA maximum-intensity projection, threshold segmentation, centreline graph coloured by
flow, inlets and outlets) beside the workflow panels (job DAG by level with the critical path, Gantt chart of Snakemake
against the tensor engine, wall-clock bars, the equivalence table, the threshold gradient and the collateral test).
Also the interactive 3D scenes: the anatomy (segmentation and centrelines by radius) and the flow sweep (wall
shear stress, idealised axial streamlines, particles, inlets and outlets)."""

from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection

from problems._common.plotting import ACCENT  # noqa: F401  (white theme and Agg backend)
from problems.medical.aneurysm_cfd.plot import boundary_glyphs, save_gif_shared_palette, write_vtp

from . import equation as eq
from . import tensors as T

MM = 1e3
GREEN, MAGENTA = "#40e65a", "#f24dd9"  # inlet / outlet colours of the aneurysm problem's glyphs
Q_MAX = 6.0  # flow colour scale (ml/s), fixed over cases, conditions and frames
RULE_COLOURS = {"segment": "#2563eb", "centreline": "#0891b2", "measure": "#16a34a", "boundaries": "#65a30d",
                "flow": "#ea580c", "report": "#9333ea", "cohort": "#db2777", "all": "#374151"}  # fmt: skip


def _case_panel(ax, img, mask, g, b, q, title, norm, cmap):
    """Axial MIP (x → patient left, y → posterior) with the segmentation and the centreline graph coloured by |Q|."""
    X, Y, _ = img.shape
    ext = [-0.5 * T.H * MM, (X - 0.5) * T.H * MM, (Y - 0.5) * T.H * MM, -0.5 * T.H * MM]
    ax.imshow(img.max(2).T, cmap="gray", vmin=0, vmax=1.2, extent=ext, interpolation="nearest")
    seg = np.ma.masked_where(~mask.any(2).T, np.ones((Y, X)))
    ax.imshow(seg, cmap=matplotlib.colors.ListedColormap(["#38bdf8"]), alpha=0.28, extent=ext, interpolation="nearest")
    ok = g["edge_valid"]
    poly = g["poly"][ok].numpy() * MM  # (E, P, 3)
    segs = [p[~np.isnan(p[:, 0])][:, :2] for p in poly]
    lc = LineCollection(segs, cmap=cmap, norm=norm, linewidths=1.0 + 1.6 * np.clip(b["r"], 0, 2.5), capstyle="round")
    lc.set_array(np.abs(q[ok.numpy()]))
    ax.add_collection(lc)
    pos = g["node_pos"].numpy() * MM
    for mask_n, col, mk in ((b["inlet"], GREEN, "^"), (b["outlet"], MAGENTA, "o")):
        p = pos[mask_n]
        ax.scatter(p[:, 0], p[:, 1], s=34, c=col, marker=mk, edgecolors="#111827", linewidths=0.6, zorder=5)
    ax.set_xlim(ext[0], ext[1]), ax.set_ylim(ext[2], ext[3])
    ax.set_xlabel(r"$x$ → patient left (mm)", fontsize=10), ax.set_ylabel(r"$y$ → posterior (mm)", fontsize=10)
    ax.tick_params(labelsize=10)
    ax.grid(False)
    t = ax.set_title(title, fontsize=11)
    return lc, t


def animate(store, dag, cases, timing, equiv, grad, path: Path):
    """store: the tensor engine's outputs; timing: per-job Snakemake times and the engine's rule times; equiv: rows of
    the equivalence table; grad: autograd and finite-difference derivatives per case."""
    B = len(cases)
    C = len(T.CONDITIONS)
    img = store["image"].numpy()
    mask = store["segment"]["mask"].numpy()
    g = store["centreline"]
    bnd = store["boundaries"]
    q = store["flow"]["flow"]["q"].detach().numpy().reshape(B, C, -1) * 1e6  # (B, C, E) ml/s
    r_e = store["measure"]["radius"].detach().numpy() * MM
    rep = {k: v.detach().numpy().reshape(B, C) for k, v in store["report"].items()}
    cmap = plt.get_cmap("plasma", 12)
    norm = matplotlib.colors.Normalize(0, Q_MAX)

    fig = plt.figure(figsize=(17.5, 12.6))
    gs = fig.add_gridspec(3, 5, width_ratios=[1, 1, 1, 0.045, 1.15], hspace=0.42, wspace=0.34)
    lcs, titles = [], []
    for k in range(min(B, 6)):
        ax = fig.add_subplot(gs[k // 3, k % 3])
        gk = {kk: v[k] for kk, v in g.items()}
        bk = {
            "inlet": bnd["inlet"][k].numpy(),
            "outlet": bnd["outlet"][k].numpy(),
            "r": r_e[k][gk["edge_valid"].numpy()],
        }
        lc, t = _case_panel(ax, img[k], mask[k], gk, bk, q[k, 0], "", norm, cmap)
        lcs.append(lc), titles.append(t)
    cax = fig.add_subplot(gs[0:2, 3])
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax)
    cb.set_label(r"centreline flow $|Q_e|$ (ml/s)", fontsize=10)
    cax.tick_params(labelsize=10)

    # job DAG drawn by level, critical path (from Snakemake's 1-core job times) in red
    ax_d = fig.add_subplot(gs[0, 4])
    jobs, A = dag["jobs"], dag["A"].numpy()
    lev = timing["levels"]
    ypos = np.zeros(len(jobs))
    for L in range(lev.max() + 1):
        idx = np.nonzero(lev == L)[0]
        ypos[idx] = np.linspace(-1, 1, len(idx)) * min(1.0, len(idx) / 12) if len(idx) > 1 else 0.0
    cp = set(timing["critical"])
    for i, j in zip(*np.nonzero(A)):
        on = i in cp and j in cp
        ax_d.plot([lev[j], lev[i]], [ypos[j], ypos[i]], color="#dc2626" if on else "#d1d5db", lw=2.2 if on else 0.6,
                  zorder=2 if on else 1)  # fmt: skip
    cols = [RULE_COLOURS[j[0]] for j in jobs]
    nodes = ax_d.scatter(lev, ypos, s=46, c=cols, edgecolors="#111827", linewidths=0.5, zorder=3)
    for name, col in RULE_COLOURS.items():
        ax_d.scatter([], [], s=30, c=col, label=name)
    ax_d.legend(loc="upper center", fontsize=9, ncol=4, frameon=False, handletextpad=0.1, columnspacing=0.5)
    ax_d.set_xlim(-0.4, lev.max() + 0.4), ax_d.set_ylim(-1.2, 1.75)
    ax_d.set_xlabel(r"level $\ell$ (dimensionless)", fontsize=10)
    ax_d.set_ylabel("job within its level (dimensionless)", fontsize=10)
    ax_d.set_yticks([]), ax_d.tick_params(labelsize=10), ax_d.grid(False)
    ax_d.set_title(f"job DAG: {len(jobs)} jobs, {int(A.sum())} edges; red = critical path", fontsize=11)

    # Gantt chart: Snakemake (12 cores, measured) and the tensor engine (one batched call per rule, measured live)
    ax_g = fig.add_subplot(gs[1, 4])
    sm = timing["snakemake12"]
    span = max(r["end"] for r in sm)
    order = sorted(range(len(sm)), key=lambda k: sm[k]["start"])
    for row, k in enumerate(order):
        r = sm[k]
        ax_g.barh(row + 9, r["end"] - r["start"], left=r["start"], color=RULE_COLOURS[r["rule"]], height=0.8)
    for row, (name, (a, b)) in enumerate(timing["engine"].items()):
        if name in RULE_COLOURS:
            ax_g.barh(row % 8, max(b - a, span / 300), left=a, color=RULE_COLOURS[name], height=0.8)
    eng_end = max(b for _, b in timing["engine"].values())
    ax_g.axhline(8.0, color="#9ca3af", lw=0.8)
    ax_g.text(span * 0.99, 3.5, f"tensor engine: {eng_end:.1f} s", ha="right", va="center", fontsize=9)
    ax_g.text(span * 0.02, 9 + len(sm) * 0.85, f"Snakemake, 12 cores: {timing['wall12']:.1f} s", ha="left",
              va="center", fontsize=9)  # fmt: skip
    cursor = ax_g.axvline(0, color="#f59e0b", lw=2)
    ax_g.set_xlim(0, timing["wall12"] * 1.02), ax_g.set_ylim(-1, len(sm) + 10)
    ax_g.set_xlabel(r"wall-clock time $t$ (s)", fontsize=10), ax_g.set_ylabel("job (dimensionless)", fontsize=10)
    ax_g.set_yticks([]), ax_g.tick_params(labelsize=10)
    ax_g.set_title("Gantt: Snakemake jobs vs tensor rules", fontsize=11)

    # wall-clock bars (log scale)
    ax_t = fig.add_subplot(gs[2, 0])
    bars = timing["bars"]
    names, vals = list(bars), list(bars.values())
    colours = ["#6b7280", "#6b7280", "#6b7280", "#d1d5db", "#16a34a", "#2563eb"][: len(vals)]
    ax_t.barh(range(len(vals)), vals, color=colours)
    for k, v in enumerate(vals):
        ax_t.text(v * 1.08, k, f"{v:.1f} s", va="center", fontsize=9)
    ax_t.set_yticks(range(len(vals)), names, fontsize=9.5)
    ax_t.invert_yaxis()
    ax_t.set_xscale("log"), ax_t.set_xlim(0.5, max(vals) * 4)
    ax_t.set_xlabel("wall-clock time for the whole cohort (s)", fontsize=10), ax_t.tick_params(labelsize=10)
    ax_t.set_ylabel("how it was run", fontsize=10)
    ax_t.set_title("time: same work, different orchestration", fontsize=11)

    # equivalence table
    ax_e = fig.add_subplot(gs[2, 1])
    ax_e.axis("off")
    cells = [[r["check"], r["tensor"], r["snakemake"], "✓" if r["ok"] else "✗"] for r in equiv]
    tab = ax_e.table(cellText=cells, colLabels=["check", "tensors", "Snakemake", ""], loc="center",
                     colWidths=[0.40, 0.26, 0.30, 0.07], cellLoc="left")  # fmt: skip
    tab.auto_set_font_size(False), tab.set_fontsize(9.5), tab.scale(1, 1.55)
    for (rr, cc), cell in tab.get_celld().items():
        cell.set_edgecolor("#e5e7eb")
        if cc == 3 and rr > 0:
            cell.get_text().set_color("#16a34a" if equiv[rr - 1]["ok"] else "#dc2626")
    ax_e.set_title("tensor engine ≡ Snakemake", fontsize=11)

    # gradient of the left-MCA flow share with respect to the threshold, autograd vs finite differences
    ax_r = fig.add_subplot(gs[2, 2])
    xk = np.arange(B)
    ax_r.bar(xk, grad["autograd"], color="#2563eb", width=0.6, label="autograd (one backward pass)")
    ax_r.scatter(xk, grad["fd"], color="#dc2626", marker="_", s=260, linewidths=2.5, zorder=3,
                 label="finite differences (2 reruns)")  # fmt: skip
    ax_r.axhline(0, color="#9ca3af", lw=0.8)
    ax_r.set_xticks(xk, cases, rotation=30, fontsize=10)
    lim = 1.25 * max(np.abs(grad["autograd"]).max(), np.abs(grad["fd"]).max())
    ax_r.set_ylim(-lim, lim * 2.0)
    ax_r.set_xlabel("case", fontsize=10)
    ax_r.set_ylabel(r"$\partial s_{\mathrm{L\,MCA}}/\partial\theta$ (1/a.u.)", fontsize=10)
    ax_r.legend(loc="upper left", fontsize=9, frameon=False)
    ax_r.tick_params(labelsize=10)
    ax_r.set_title(r"gradient through the pipeline ($s$ = L-MCA share)", fontsize=11)

    # collateral test: left-hemisphere outflow with the left ICA closed, as % of baseline
    ax_c = fig.add_subplot(gs[2, 3:5])
    keys = [("Q_L-MCA", "L-MCA"), ("Q_L-ACA", "L-ACA"), ("Q_L-PCA", "L-PCA")]
    w = 0.26
    for j, (key, lab) in enumerate(keys):
        base, occl = rep[key][:, 0], rep[key][:, 1]
        pct = np.where(base > 1e-9, 100 * occl / np.maximum(base, 1e-12), np.nan)
        ax_c.bar(xk + (j - 1) * w, pct, width=w, color=ACCENT[j], label=lab)
    ax_c.axhline(100, color="#9ca3af", lw=0.8, ls="--")
    ax_c.set_ylim(0, 128)
    ax_c.set_xticks(xk, cases, rotation=30, fontsize=10)
    ax_c.set_xlabel("case", fontsize=10), ax_c.set_ylabel("outflow, L-ICA closed / open (%)", fontsize=10)
    ax_c.legend(loc="upper center", fontsize=9, ncol=3, frameon=False)
    ax_c.tick_params(labelsize=10)
    ax_c.set_title("collateral test: left ICA inlet closed", fontsize=11)

    n_frames = 24

    def update(f):
        c = 0 if f < n_frames // 2 else 1
        for k in range(len(lcs)):
            ok = g["edge_valid"][k].numpy()
            lcs[k].set_array(np.abs(q[k, c][ok]))
            cond = "baseline" if c == 0 else "L-ICA closed"
            titles[k].set_text(f"{cases[k]} · Dice {rep['dice'][k, 0]:.2f} · {cond}\n"
                               f"Δp {rep['dp_max_mmHg'][k, c]:.1f} mmHg, "
                               f"L-MCA {rep['Q_L-MCA'][k, c]:.2f} ml/s")  # fmt: skip
        t = timing["wall12"] * f / (n_frames - 1)
        cursor.set_xdata([t, t])
        done = np.array([0.0] * len(jobs))
        for k, j in enumerate(jobs):
            e = timing["end12"].get(j)
            done[k] = 1.0 if (e is not None and e <= t) or (e is None and f == n_frames - 1) else 0.25
        nodes.set_alpha(None)
        rgba = matplotlib.colors.to_rgba_array(cols)
        rgba[:, 3] = done
        nodes.set_facecolor(rgba)

    fig.suptitle("Circle of Willis cohort (TopCoW MRA, CC0) as a tensor DAG — the same Snakefile run by Snakemake "
                 "and by tensors; educational flow model, not for clinical use", fontsize=12)  # fmt: skip
    fig.subplots_adjust(left=0.095, right=0.985, top=0.925, bottom=0.07)
    return save_gif_shared_palette(fig, update, list(range(n_frames)), path, fps=3)


# ---------------------------------------------------------------- interactive 3D scene (VTK collection)
def _disc(centre, normal, r, m=16):
    """A flat fan of triangles (m, 3, 3) in the plane ⟂ normal, wound so its area vector points along normal."""
    n = normal / np.linalg.norm(normal)
    u = np.cross(n, [1.0, 0, 0] if abs(n[0]) < 0.9 else [0, 1.0, 0])
    u /= np.linalg.norm(u)
    v = np.cross(n, u)
    t = np.linspace(0, 2 * np.pi, m + 1)
    rim = centre + r * (np.cos(t)[:, None] * u + np.sin(t)[:, None] * v)
    return np.stack([np.repeat(centre[None], m, 0), rim[:-1], rim[1:]], 1)


def _slices(img, offset):
    """Three orthogonal planes through the crop centre as triangle grids at 1 mm, with the MRA intensity per vertex."""
    pts, tris, vals = [], [], []
    X, Y, Z = img.shape
    for axis in range(3):
        k = img.shape[axis] // 2
        plane = np.take(img, k, axis=axis)[::2, ::2]
        a, b = plane.shape
        ia, ib = np.meshgrid(np.arange(a) * 2, np.arange(b) * 2, indexing="ij")
        coords = [None, None, None]
        rest = [d for d in range(3) if d != axis]
        coords[axis], coords[rest[0]], coords[rest[1]] = np.full_like(ia, k), ia, ib
        P = np.stack([c.ravel() for c in coords], 1) * T.H * MM + offset
        base = sum(len(p) for p in pts)
        idx = np.arange(a * b).reshape(a, b)
        q = np.stack([idx[:-1, :-1].ravel(), idx[1:, :-1].ravel(), idx[1:, 1:].ravel(), idx[:-1, 1:].ravel()], 1)
        tris.append(np.concatenate([q[:, [0, 1, 2]], q[:, [0, 2, 3]]]) + base)
        pts.append(P), vals.append(plane.ravel())
    return np.concatenate(pts), np.concatenate(tris), np.concatenate(vals)


def decimate(V: np.ndarray, F: np.ndarray, cell: float) -> tuple[np.ndarray, np.ndarray]:
    """Vertex clustering: vertices in the same cell (m) merge at their mean; degenerate and repeated triangles go."""
    _, inv = np.unique(np.floor(V / cell).astype(np.int64), axis=0, return_inverse=True)
    inv = inv.ravel()
    n = inv.max() + 1
    cnt = np.bincount(inv, minlength=n)
    Vc = np.stack([np.bincount(inv, V[:, i], n) / cnt for i in range(3)], 1)
    Fc = inv[F]
    Fc = Fc[(Fc[:, 0] != Fc[:, 1]) & (Fc[:, 1] != Fc[:, 2]) & (Fc[:, 0] != Fc[:, 2])]
    _, first = np.unique(np.sort(Fc, axis=1), axis=0, return_index=True)
    return Vc, Fc[np.sort(first)]


def _collection(path: Path, entries: list[str]) -> Path:
    head = '<?xml version="1.0"?>\n<VTKFile type="Collection" version="1.0" byte_order="LittleEndian"><Collection>\n'
    path.write_text(head + "\n".join(entries) + "\n</Collection></VTKFile>\n", encoding="ascii")
    return path


RINGS = ((0.0, 1), (0.35, 3), (0.7, 6))  # streamline seeds per vessel: (ρ = distance from the axis / r, angles)
DT = 4e-3  # pseudo-time step (s) the particles advance by between two sweep steps (each step is a steady state)
SEED_SPACING = 3.0  # one particle per this many mm of streamline
STREAM_STEP = 1.5  # streamline point spacing (mm)


def resample(pe: np.ndarray, step: float) -> np.ndarray:
    """A polyline (n, 3) resampled at equal arc length, about `step` apart, end points kept."""
    s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pe, axis=0), axis=1))]
    n = max(int(np.ceil(s[-1] / step)), 1) + 1
    t = np.linspace(0, s[-1], n)
    return np.stack([np.interp(t, s, pe[:, i]) for i in range(3)], 1)


def transport_frame(pe: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Orthonormal normals (n, 3), binormals (n, 3) along a polyline by parallel transport: each normal is the previous
    one with its component along the new tangent removed, so the frame does not twist around the axis."""
    t = np.gradient(pe, axis=0)
    t /= np.maximum(np.linalg.norm(t, axis=1, keepdims=True), 1e-12)
    n = np.zeros_like(pe)
    a = np.cross(t[0], [1.0, 0, 0] if abs(t[0, 0]) < 0.9 else [0, 1.0, 0])
    n[0] = a / np.linalg.norm(a)
    for i in range(1, len(pe)):
        a = n[i - 1] - np.dot(n[i - 1], t[i]) * t[i]
        nrm = np.linalg.norm(a)
        n[i] = a / nrm if nrm > 1e-9 else n[i - 1]
    return n, np.cross(t, n)


def axial_streamlines(pe: np.ndarray, r: float, rings=RINGS) -> tuple[np.ndarray, np.ndarray]:
    """Streamlines of axial Poiseuille flow in a tube of radius r around the polyline pe (n, 3): curves parallel to
    the centreline at ρ r from it (ρ, angle from `rings`). Returns points (S, n, 3) and ρ (S,)."""
    nrm, bin_ = transport_frame(pe)
    lines, rho = [], []
    for rh, m in rings:
        for phi in 2 * np.pi * (np.arange(m) + 0.5 * (rh > 0.5)) / m:
            lines.append(pe + rh * r * (np.cos(phi) * nrm + np.sin(phi) * bin_))
            rho.append(rh)
    return np.stack(lines), np.array(rho)


def along(line: np.ndarray, s: np.ndarray) -> np.ndarray:
    """Points at arc lengths s (wrapped into [0, length)) along a polyline (n, 3)."""
    c = np.r_[0, np.cumsum(np.linalg.norm(np.diff(line, axis=0), axis=1))]
    s = np.mod(s, max(c[-1], 1e-9))
    return np.stack([np.interp(s, c, line[:, i]) for i in range(3)], 1)


def write_scene(folder: Path, store, cases, surfaces, sev, sweep) -> tuple[Path, Path]:
    """Two 3D results, per case tiled 3 × 2 (60 mm apart):
    centreline.pvd — the anatomy pipeline: the segmentation surface and the centreline graph, both coloured by the
      vessel radius (static);
    flow.pvd — the flow of the 1D Poiseuille network, one timestep per left-ICA stenosis severity s (0 open … 1
      closed): the vessel surface coloured by the wall shear stress 4μ|Q|/(πr³) of the nearest centreline edge (and
      its pressure), idealised axial streamlines per vessel (speed 2U(1 − ρ²)), particles moved along them by a
      fixed pseudo-time DT per step, and the inlet (green) and outlet (magenta) discs with arrows."""
    folder.mkdir(parents=True, exist_ok=True)
    B, C = len(cases), len(sev)
    g, bnd, m = store["centreline"], store["boundaries"], store["measure"]
    q = np.stack([fs["q"].detach().numpy() for fs in sweep], 1)  # (B, S, E) m³/s, u → v
    p = np.stack([fs["p"].detach().numpy() for fs in sweep], 1) / T.MMHG  # (B, S, N) mmHg
    steps = [f"{float(s):.2f}" for s in sev]
    r_m = m["radius"].detach().numpy()  # (B, E) m
    r_e = r_m * MM
    anatomy = []
    groups = {name: [] for name in ("inlet", "outlets")}
    per_t = {c: {"surface": [], "streamlines": [], "particles": []} for c in range(C)}
    for k in range(B):
        offset = np.array([60.0 * (k % 3), 55.0 * (k // 3), 0.0])
        ok = g["edge_valid"][k].numpy()
        poly = g["poly"][k].numpy() * MM
        E = int(ok.sum())
        lines_pts, lines_idx, eid, frac = [], [], [], []
        n0 = 0
        for e in range(E):
            pe = poly[e][~np.isnan(poly[e][:, 0])]
            s = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pe, axis=0), axis=1))]
            lines_pts.append(pe + offset), lines_idx.append(np.arange(n0, n0 + len(pe)))
            eid.append(np.full(len(pe), e)), frac.append(s / max(s[-1], 1e-9))
            n0 += len(pe)
        LP, EI, FR = np.concatenate(lines_pts), np.concatenate(eid), np.concatenate(frac)
        u, v = g["edges"][k, :E, 0].numpy(), g["edges"][k, :E, 1].numpy()
        V, F = decimate(*surfaces[k], cell=1.0e-3)  # 1 mm clusters: 11 flow steps stay light
        Vmm = V * MM + offset
        Vt, Lt = torch.tensor(Vmm), torch.tensor(LP)
        near = torch.cat([torch.cdist(Vt[i : i + 8192], Lt).argmin(1) for i in range(0, len(Vt), 8192)]).numpy()
        # anatomy (static): segmentation surface and centrelines coloured by radius
        fa = write_vtp(folder / f"centreline_{cases[k]}.vtp", LP, lines=lines_idx,
                       point_data={"radius": (r_e[k][EI], "mm")})  # fmt: skip
        fs = write_vtp(folder / f"segmentation_{cases[k]}.vtp", Vmm, polys=F,
                       point_data={"radius": (r_e[k][EI][near], "mm")})  # fmt: skip
        anatomy += [f'<DataSet timestep="0" group="segmentation" part="1" file="{fs.name}"/>',
                    f'<DataSet timestep="0" group="centrelines" part="2" file="{fa.name}"/>']  # fmt: skip
        # streamline geometry (fixed) and particle seeds (fixed, deterministic) per vessel edge
        rng = np.random.default_rng(k)
        sl, sl_edge, sl_rho, seeds = [], [], [], []
        for e in range(E):
            pe = resample(poly[e][~np.isnan(poly[e][:, 0])], STREAM_STEP) + offset
            X, rho = axial_streamlines(pe, r_e[k, e])
            for line, rh in zip(X, rho):
                c_len = float(np.linalg.norm(np.diff(line, axis=0), axis=1).sum())
                n_p = max(int(c_len / SEED_SPACING), 1)
                sl.append(line), sl_edge.append(e), sl_rho.append(rh)
                seeds.append((np.arange(n_p) + rng.uniform()) * c_len / n_p)
        sl_edge, sl_rho = np.array(sl_edge), np.array(sl_rho)
        sl_pts = np.concatenate(sl)
        n_per = np.array([len(x) for x in sl])
        sl_idx = np.split(np.arange(len(sl_pts)), np.cumsum(n_per)[:-1])
        n_seed = np.array([len(x) for x in seeds])
        shift = np.zeros(len(sl))  # particle displacement along each streamline so far (mm)
        for c in range(C):
            pl = p[k, c][u][EI] * (1 - FR) + p[k, c][v][EI] * FR  # pressure linear along each edge
            wss = eq.wall_shear_stress(q[k, c, :E], r_m[k, :E], T.MU)  # (E,) Pa
            f1 = write_vtp(folder / f"surface_{cases[k]}_t{c}.vtp", Vmm, polys=F,
                           point_data={"WSS": (wss[EI][near], "Pa"), "pressure": (pl[near], "mmHg")})  # fmt: skip
            U = np.abs(q[k, c, :E]) / (np.pi * r_m[k, :E] ** 2)  # mean speed per edge (m/s)
            spd = eq.poiseuille_speed(sl_rho, U[sl_edge])  # (S,) constant along each streamline
            f2 = write_vtp(folder / f"streamlines_{cases[k]}_t{c}.vtp", sl_pts, lines=sl_idx,
                           point_data={"speed": (np.repeat(spd, n_per), "m/s")})  # fmt: skip
            if c > 0:  # a fixed pseudo-time step at this step's speed, in the direction of this step's flow
                shift += np.sign(q[k, c, :E])[sl_edge] * spd * DT * MM
            P = np.concatenate([along(line, s0 + d) for line, s0, d in zip(sl, seeds, shift)])
            f3 = write_vtp(folder / f"particles_{cases[k]}_t{c}.vtp", P, verts=True,
                           point_data={"speed": (np.repeat(spd, n_seed), "m/s")})  # fmt: skip
            for name, f in (("surface", f1), ("streamlines", f2), ("particles", f3)):
                per_t[c][name].append(f)
        # boundary glyphs exactly as in the aneurysm problem: a disc in the cut plane and an arrow along the flow
        pos = g["node_pos"][k].numpy() * MM + offset
        nrm = bnd["normal"][k].numpy()
        be = bnd["edge"][k].numpy()
        for kind, mask in (("inlet", bnd["inlet"][k].numpy()), ("outlets", bnd["outlet"][k].numpy())):
            for n in np.nonzero(mask & (be >= 0))[0]:
                disc = _disc(pos[n], nrm[n], max(r_e[k][be[n]], 0.4))
                tris = torch.tensor(disc / MM)
                ids = torch.full((len(disc),), 1 if kind == "inlet" else 2)
                gl = boundary_glyphs(tris, ids, LP)
                groups[kind].append(gl["inlet" if kind == "inlet" else "outlets"])
    entries = []
    for c in range(C):
        for part, (name, files) in enumerate(per_t[c].items()):
            for f in files:
                entries.append(f'<DataSet timestep="{steps[c]}" group="{name}" part="{part}" file="{f.name}"/>')
    for part, (name, colour) in enumerate((("inlet", GREEN), ("outlets", MAGENTA)), start=3):
        pts, tri, n0 = [], [], 0
        for P_, T_ in groups[name]:
            pts.append(P_), tri.append(T_ + n0)
            n0 += len(P_)
        f = write_vtp(folder / f"{name}.vtp", np.concatenate(pts), polys=np.concatenate(tri))
        for c in range(C):
            entries.append(
                f'<DataSet timestep="{steps[c]}" group="{name}" part="{part}" file="{f.name}" color="{colour}"/>'
            )
    return _collection(folder / "centreline.pvd", anatomy), _collection(folder / "flow.pvd", entries)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'threshold': 0.3,
    'cores': 12
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.cow_workflow_dag")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.cow_workflow_dag.main")
problem_dir = Path("problems/medical/cow_workflow_dag")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
